<a href="https://colab.research.google.com/github/admmuaka/admmuaka.github.io/blob/main/analyse_textuelle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traitement du langage: analyse textuelle

---

1. Import packages
2. Exemple de phrase brute
3. Nettoyage
4. Données d'entraînement
5. Entraînement du modèle
6. Extration d'entités

---

## 1. Import packages

In [3]:
import pandas as pd
import re
import spacy
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
from spacy.training import offsets_to_biluo_tags




## 2. Exemple de phrase brute

In [9]:
raw_text = """
1. PLUNGER # 60= 2pcs _x000D_
2.SEAL RUBBER #130=1pc._x000D_
3.COUPLER  #430=2pcs. _x000D_
4.STRAPS# 360=1pc. _x000D_
5.DUCT FLEXIBLE #70=1pc_x000D_
6.KIT CLAMP #20= 1pc._x000D_
7.SCREW SOCKET HEAD CAP #110= 2pcs._x000D_
8.FLAT WASHER #210=2pcs._x000D_
9. HOSE (25 FT) #50=1pc._x000D_
10.COVER DRAIN #20=1pc.
#11.ITEM P044 PLUMB LINE P/N: 98L21508593.044 x2.
11.Item 130
12.ITEM # -12 WEB SLING x1. ITEM#-26 BOLT = x1
13.1. AIR CHUCK ITEM-6 = x1. 2. HOSE ASSY ITEM-39 = x1. 3. NEEDLE VALVE ITEM-4 = x1.
14.A LOAD ASSY -58 "E" M83723/60-18AR x1. -2 ASSY 7101SYCQE 89 & 91 x1.
15.0-16 BAR GAUGE x1.
16.1. FLOURESCENT TUBE 48" x4
17.1. CASTERING WHEEL P21 x2 2. CASTERING WHEEL "STOP FIX" P25 x2
18.1. WING THUMB NUT-x1. 2. DIAL INDICATOR CABLE = x1. 3. EXTENSION CABLE = x1. # séparer NUT de -x1  phrase 300
19.1. WING THUMB NUT-x1. 2. DIAL INDICATOR CABLE = x1. 3. EXTENSION CABLE = x1. 4. P09-x1
20.1. TENSIONER PAD P22 x1. 2. CABLE ASSEMBLY P26 -x1.
21.ITEM P532-NON
22.ITEM P23 HAMMER DRIVE x4. ITEM P25 MARKER x1.
23.Below items to replace: P6 /8 QT1 P10 QT4 P11 QT8 P9 QT4
24.1-Streamer to be changed
25.text = "1. MS3181-10RAL-6 qté1 missing"
26.To be replaced: 1- P7 : quantity 4 2- P2/15/16: Quantity 1 3- P11.2 Quantiy 6
27.tata P2/15
28 Air chuck to be replaced -item6
29.1-FOO and 1870-14-14
30.1. CLUTCH SPANNER PN:98F07203020208
31.2 QTY OF FRICTION DISC CHANGE 1 QTY CHAIN ROLL OFF ELEMENT CHANGE
32.AP272-K-06-A
33.6. DOWEL PIN ITEM30 x12 and -ITEM12 and 1.pc
34.INDICATOR BATTERY QTY:2
35.REPLACE BATTERY, 2 QTY
36."1-Hose assemblies damages to be replaced 2-Handpump assembly to be replaced
37.1-FOO and 1870-14-14 and AP272-K-06-A , 2-Hose assemblies damages to be replaced 3-Handpump assembly to be replaced
38.1-FOO and 1870-14-14 and AP272-K-06-A , 2-hose assemblies damages to be replaced 3-Handpump assembly to be replaced 7-tata 8-TATA 9-TaTA 10-Tata
40.VALVE-6 should be cleaned. AP272-K-06-A is excluded. and 1-F00 and P254-V-10
41.PARTS NEED REPALCEMENT P254, 262 AND 284.
42.ITEM NO: 2 = x1 , ITEM NO. 4 - x2 , ITEM NO.4 - x2 , 1. ROD END ITEM NO: 4 = x1 , 1. SCREW ITEM No. 906 x4
43.1. PROTECTIVE RUBBER PADS P301-1, 301-2, 401-1 AND 401-2
44.4 items 10 : STUD 8 items 11 : PINS , ITEM20
45.1. NYLON WASHER ITEM = 15. x3
46.PROTECTIVE RUBBER PADS P301-1, 301-2, 401-1 AND 401-2
47.PIN P08, P09 & 10 = x1. each 2. PLASTIC CLIP x4
48.P8-CUSHION/RUBBER P31-BOLT x6. P32-ADHESIVE x1
49.1. ROUND P314 x2 2. ROUNDITEM 316 x2 3. NUT TURN BUCKLE P322 x2 4. ROUND P324 x2 5. SPRING TYPE STRAIGHT PIN P726 x2
50.P02 & 3
51.ITEMS P258-GUARD RAIL 260. ROD 498. TUBE 046- MEASURING DEVICE
52.7ITEMP18
53.1 manometer 0-4bars 1 washer NFE27611 Rubber to replace on items -44 & -46
54.WINCH x1 11- 12-& 13- CABLE x1
55.INBD AFT RAIL ASSYP2
56. P29, P31, 39 x2 Each 6. DUST PLUG P32, P22, 40 x3 Each 7. O-RING P26, P34 x2 Each
57.CABLE P39 & -40 KEENSERT
58.1. components Missing P6, P7, 8,9, 19,20 & 21) 2. Pressure Regulator = x1.
59.iterm-16
60.ITEM NOP8 GAUGE x1 -15 BLEED PORT ADAPTER O-RING x1
61.1)-56 SPACING WASHER 2) -43 SLEEVE, 3) -50 ROD END, 4) -51 JAM NUT,5) -52 PLAIN WASHER , 6)-129 HAND KNOB ASSY 7) -125 SPRING PIN. 8) K71001-80 AFT HAND KNOB x2, 9)K71001-104 (SXWP3) = x1 10. P45, P46 x2 each
62.ASSY# J32002-94 x1
63.P9/-23 CONNECTOR
64.ITEM#- -25 P14 LOWER
65.ITEM #: KG -20-1 ROD END. LH = x1. ITEM #: -9 JAM NUT, LH x2 ITEM #: -10 JAM NUT, RH = x1.
66.DUST PLUG P32, P22, 40
67.P6, P7, P8, P9, 19,20 & 21)
68. ITEM P05 CAPTIVE SCREW > 1 ASSY
69.13, 28,30 & 12  and -32 & - 31
70.P84-NUT THIN x1, P85-WASHER SPRING S/COIL x1.
71.item: 25 nut hex M6 24 locating pin
72. $620.00 , ${2, 300.00}USD , $1,060
73.A27092-16;-18 BLOCK
74.ASSY#2 , GAUGE x1 -34CAP
75.TEMP33 SCREW PIN ANCHOR SHACKLE and ITEMP33 SCREW PIN ANCHOR SHACKLE
76.ASSY x1 DAMAGED ON -32 & - 31 DEACT/ACTUAT ASSY x1 x1
77.Placard-70, 89 &85   and 956A8304P21,PAWL and AFP241-24 and P707-SLEEVE-ASSY and #956A8667P11-X1
78. P20 35420€ tete
79.Rail Qty (1) PCS -9C1165P11, HEX HEAD NUT, QTY (2) PCS and (2) and(x2) and P8(x1)%
80.CSTDVIS0852* MISSING
81.2(2) and P26(x2) and -6(2) and (2027) and (2005)
82.P53x1° P166x2° - NO QUOTE P48x8# P54x2# P44x4# P113x1° P189Ax1° P66x1# P74x1# P73x2# P105x1# P173x1# P46x1#
83.£49.00 Fitting
84.Detail 29;Details 30; Det 01 ;P21->Rotation and P/N-15
85.1/ 4x retaining pins 2/ 4x bearings Cost €675
86.4X -78 washer and qty-1
87. X1 AXE X2
88.Detail06
89.Discard items: -24/-23/-22/-21/-20 2、Add items: -10/-11/-12/-13/-14 3 and  P44/-46
90.-ITEM -30,-31, 32,33, 34,35, 36,37, 40
91.-ITEMs -101,-247 AND -248
92.-704 ITEM
93.95 X1PC -8 X1PC -34 X1PCS -132 X2PCS -60 X2PCS -56 X1PCS -115 X22PCS -116 X22PCS -68 X1PC
94.1/ Details 02 and 03 modified. 2/ Remove Detail 45. 3/ Add Detail 85 to 94.
95. p10x2p11x1 and P20=x1 , P20 =x1, P20= x1 , P20 = x1
96.P38 et P20  , P20 et 20 = x1 , 20QT2 20QT 2
97.24/23/22/21/20/21
98.6(x2), -6(x2) ,  P6(x2) , -P32(x2)
99.856A3576P02(A,B,C,D)x1
100.AM -25663P225  , P225B
101.P20 =20  ; P722 x4 & 726 x1 ; P63 x1,64 x1
102.P35,-36, -37
103.QT:1    20 - x2
104.P24 x2 / 27x4 /24 x4 / 22 x2 /21 x2/ 21x2 /20 x2
105.P51: x2  and Pin: x2 and Pin, x2
106.P20 rep A -> P20-A
107.qt = 2
108.Assy 202 and 302
109.Part 93 COMBINED BEARING x9 Part 200 SHOCK ABSORBER x22
110.P14-15-16-17; -13ITEMP59; P11-21-33; REP9-19-15; P7-SET; P32-A-B-C; P7-14-9-11



"""




## 3. Nettoyage

In [4]:

def nettoyer_description(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("_x000D_", " ")
    text = re.sub(r'(\b\w+)\(([^)]+)\)x(\d+)', lambda m: ', '.join(f'{m.group(1)}-{ch} x{m.group(3)}' for ch in m.group(2).split(',')), text) #856A3576P02(A,B,C,D)x1
    text = re.sub(r'\b(ROUND)(ITEM)\b', r'\1 \2', text, flags=re.IGNORECASE) #ROUND ITEM
    text = re.sub(r'\b(_Detail)(\d+)\b', r'\1 \2', text, flags=re.IGNORECASE) # Detail 06
    text = re.sub(r'iterm',r'ITEM',text)
    text = re.sub(r'\b(ASSY)(#\d+)\b', r'\1 \2', text, flags=re.IGNORECASE) #ASSY#2 -> ASSY #2
    text = re.sub(r'ASSY#',r'ASSY',text)
    text = re.sub(r'\bPart\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques " Part12", " Part  12" → P12
    text = re.sub(r'\bREP\s*(\d+)', r'P\1',text,flags=re.IGNORECASE) #REP9-19-15 -> P9-19-15
    text = re.sub(r'\b(\d+) ITEM\b', r'ITEM\1', text,flags=re.IGNORECASE)  #-704 ITEM -> -ITEM704
    text = re.sub(r'(?<!\w)(-\d+)(ITEMP59)(?!\w)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*(\d+)', r'P\1', text)
    text = re.sub(r'\bITEM\s*#\s*-\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEM#-12", "ITEM # -12" → P12
    text = re.sub(r'\bITEM\s*#\s*-\s*-\s*(\d+)', r'P\1', text, flags=re.IGNORECASE)#Cas spécifiques "ITEM#- -12", "ITEM #- -12" → P12
    text = re.sub(r'\bITEM\s*#\s*:\s*-\s*(\d+)', r'P\1', text, flags=re.IGNORECASE)#Cas spécifiques "ITEM# :12", "ITEM #:12" → P12
    text = re.sub(r'\bITEM\s*NO\s*:\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEMNO:12", "ITEM NO :12" → P12
    text = re.sub(r'\bITEM\s*NO\s*.\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEMNO.12", "ITEM NO .12" → P12
    text = re.sub(r'\bITEM\s*NO\s*#\s*-\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEMNO.12", "ITEM NO .12" → P12
    text = re.sub(r'\bITEM\s*NO\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEMNO 12", "ITEM NO .12" → P12
    text = re.sub(r'\bITEM\s*-\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEM-12", "ITEM  -12" → P12
    text = re.sub(r'\bITEMS\s*-\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEMS-12", "ITEMS  -12" → P12
    text = re.sub(r'\bITEM\s*#\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEM#12", "ITEM # 12" → P12
    text = re.sub(r'\bITEM\s*:\s*(\d+)', r'P\1', text, flags=re.IGNORECASE)#Cas spécifiques "ITEM :12", "ITEM :12" → P12
    text = re.sub(r'\bITEM\s*=\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEM=12", "ITEM = 12" → P12
    text = re.sub(r'-ITEM\s*(\d+)', r'P\1',text, flags=re.IGNORECASE) #  Cas  : "-ITEM12" ou "-ITEM 12" → P12
    text = re.sub(r'\bITEM\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEM12", "ITEM  12" → P12
    text = re.sub(r'\bITEMS\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ITEMS12", "ITEMS  12" → P12
    text = re.sub(r'\bITEMS\s*:\s*-\s*(\d+)', r'P\1',text , flags=re.IGNORECASE)#Cas spécifiques "ITEM : -12", "ITEM :-12" → P12
    text = re.sub(r'\bASSyITEM\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "ASSYITEM12", "ASSYITEM  12" → P12
    text = re.sub(r'\bDetail\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "Detail12", "Detail  12" → P12
    text = re.sub(r'\bDetails\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "Details12", "Details  12" → P12
    text = re.sub(r'\bDet\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "Details12", "Details  12" → P12
    text = re.sub(r'7ITEMP',r'P',text) # 7ITEMP -> P
    text = re.sub(r'#\s*-\s*(\d+)', r'P\1',text) #Cas génériques "#-12", "# -12" → P12
    #text = re.sub(r'-\s*(\d+)', r'P\1',text, flags=re.IGNORECASE)#Cas spécifiques "-12", → P12
    text = re.sub(r'(?i)=\s*(\d+)\s*pcs\.?', r' x\1', text,flags=re.IGNORECASE)
    text = re.sub(r'(?i)\b(\d+)\s*pcs\b', r'x\1', text,flags=re.IGNORECASE)
    text = re.sub(r'(?i)\b(\d+)\s*pc\b', r'x\1', text, flags=re.IGNORECASE)
    text = re.sub(r'(?i)\b(\d+)\s*.pc\b', r'x\1', text, flags=re.IGNORECASE) # Transforme = 2.pc en x2
    text = re.sub(r'\bQTY\s*(\d+)', r'x\1', text, flags=re.IGNORECASE) # QTY 1 -> X1
    text = re.sub(r'\bQuantity\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QTY 1 -> X1
    text = re.sub(r'\bQuantiy\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QTY 1 -> X1
    text = re.sub(r'\bqté\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QTé 1 -> X1
    text = re.sub(r'\bqt\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QT 1 -> X1
    text = re.sub(r'\bqty:\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QTY: 1 -> X1
    text = re.sub(r'\bqty :\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QTY :1 -> X1
    text = re.sub(r'\bqty-\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QTY :1 -> X1
    text = re.sub(r'\bqyt:\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # QTY: 1 -> X1
    text = re.sub(r'\b(\d+)\s*QTY\b', r'x\1', text, flags=re.IGNORECASE) #2 QTY -> x2
    text = re.sub(r'\s*QT\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # 20QT2 20QT 2  20 QT 2  20 QT2
    text = re.sub(r'\s*QT\s*:\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # 20QT:2
    text = re.sub(r'\s*QT\s*=\s*(\d+)', r'x\1',text, flags=re.IGNORECASE) # 20QT =2
    text = re.sub(r'QTY\s*\(\s*(\d+)\s*\)\s*PCS', r'x\1', text, flags=re.IGNORECASE) #♠QTY (2) PCS -> x2
    #text = re.sub(r'\(\s*(\d+)\s*\)', r'x\1', text, flags=re.IGNORECASE) #(2) -> x2
    text = re.sub(r'\(\s*(?!19\d{2}|20\d{2})(\d+)\s*\)', r'x\1', text, flags=re.IGNORECASE)#(2) -> x2 Ne pas transformer si c'est une année entre 1900 et 2099
    text = re.sub(r'(?<=\d)x(?=x?\d)', r' x', text) # 2x2 -> 2 x2
    text = re.sub(r'([A-Za-z])x(\d+)', r'\1 x\2', text) # P189Ax1 -> P189A x1
    text = re.sub(r'(\d+)(\()', r'\1 \2', text, flags=re.IGNORECASE) #06(x2) -> 06 (x2)
    text = re.sub(r'\b(\d+)[xX]\b', r'x\1', text)  #4x -> x4
    text = re.sub(r'\s*=\s*x', r' x',text,flags=re.IGNORECASE)# P20=x1 -> P20 x1
    text = re.sub(r'\s*=\s*(\d+)', r' x\1', text,flags=re.IGNORECASE) #♥ =1 ->x1
    text = re.sub(r'(?<=\S)=(?=\S)', ' = ', text)# Ajouter un espace autour du signe égal : P50=x1 -> P50 = x1
    text = re.sub(r'\s*=\s*', ' = ',text) # Ajoute espace autour de tous les signes "=" (corrige P20= x1, P20 =x1, etc.)
    text = re.sub(r'\bitem\s+(?![Pp])(\d+)', r'P\1', text, flags=re.IGNORECASE)
    text = re.sub(r'\$\s*\d{1,3}(?:,\d{3})*(?:\.\d{2})?', '', text) # 1,060
    text = re.sub(r'\$\s*\d+(?:\.\d+)?', '', text)  #$620.00 -> rien
    text = re.sub(r'\£\s*\d+(?:\.\d+)?', '',text) #£620.00 -> rien
    text = re.sub(r'\£\s*\d+(?:\,\d+)?', '',text) #£620,00 -> rien
    text = re.sub(r'\$\{[^}]*\}', '', text) #${2, 300. 00}USD
    text = re.sub(r'\s*\d+(?:\.\d+)?\€', '', text)  #620.00€ -> rien
    text = re.sub(r'\€\s*\d+(?:\.\d+)?', '', text)  #€620.00 -> rien
    text = re.sub(r'\b(\d+)\.(\S)', r'\1. \2', text)# Séparer les énumérations comme "1.FOO" → "1. FOO"
    text = re.sub(r'\b(\d+)\,(\S)', r'\1, \2', text)# Séparer les énumérations comme "1,FOO" → "1, FOO"
    text = re.sub(r'\b(\d+)\;(\S)', r'\1, \2', text)# Séparer les énumérations comme "1;FOO" → "1; FOO"
    text = re.sub(r'\b(\d+)\)(\S)', r'\1. \2',text)# 🔁 Séparer les énumérations comme "1)FOO" → "1) FOO"
    text = re.sub(r'(^|\s)(\d+)-([a-zA-Z]{2,})(?![\w-])', r'\1\2. \3', text) #Remplace uniquement les énumérations de type"1-FOO" → "1. FOO"  mais ignore les formats techniques,ne change pas 1870-14-14 ni AP272-K-06-A
    text = re.sub(r'\bITEMP(\d+)', r'ITEM P\1', text, flags=re.IGNORECASE) # Séparer "ITEMP" collé en "ITEM P"
    text = re.sub(r'\bTEMP(\d+)', r'ITEM P\1', text, flags=re.IGNORECASE) # Séparer "TEMP" collé en "ITEM P"
    text = re.sub(r'-P(\d+),\s*-\s*(\d+), \s*(\d+),\s*(\d+), \s*(\d+),\s*(\d+), \s*(\d+),\s*(\d+), \s*(\d+)', r'P\1, P\2, P\3, P\4, P\5, P\6, P\7, P\8, P\9',text) #P08, P09 & 10 -> P08, P09 P10
    text = re.sub(r'P(\d+),\s*P(\d+),\s*P(\d+),\s*P(\d+),\s*(\d+),\s*(\d+)\s*&\s*(\d+)', r'P\1, P\2, P\3,P\4, P\5, P\6,P\7',text) #P08, P09 & 10 -> P08, P09 P10
    text = re.sub(r'P(\d+)/([\d/]+)', lambda m: f"P{m.group(1)} " + " ".join(f"P{x}" for x in m.group(2).split('/')), text)#P2/15/16 -> P2 P15 P16
    #text = re.sub(r'(P\d+)(/)', r'\1 \2', text) # P9/-23 -> P9 /-23
    text = re.sub(r'\bP(\d+),\s*(\d+)(?:\s*(?:AND|et)\s*(\d+))?', lambda m: f'P{m.group(1)}, P{m.group(2)}' + (f' P{m.group(3)}' if m.group(3) else ''), text, flags=re.IGNORECASE) #P254, 262 AND 284. -> P254, P262 P284.
    text = re.sub(r'P(\d+)-(\d+),\s*(\d+)-(\d+),\s*(\d+)-(\d+)\s*AND\s*(\d+)-(\d+)', r'P\1-\2, P\1-\4, P\5-\6 P\5-\8',text) #P301-1, 301-2, 401-1 AND 401-2
    text = re.sub(r'-P(\d+),\s*-\s*(\d+)\s*AND\s*-\s*(\d+)', r'P\1, P\2, P\3', text) #-P101,-247 AND -248
    text = re.sub(r'P(\d+)\s*,\s*-\s*(\d+)\s*,\s*-\s*(\d+)', r'P\1, P\2, P\3',text) #P101,-247 ,-248
    text = re.sub(r'P(\d+)\s*and\s*(\d+)', r'P\1, P\2',text, flags=re.IGNORECASE) #P101 and 248
    text = re.sub(r'P(\d+)\s*et\s*(\d+)', r'P\1, P\2',text, flags=re.IGNORECASE) #P101 et 248
    text = re.sub(r'P(\d+),\s*P(\d+)\s*&\s*(\d+)', r'P\1, P\2, P\3', text) #P08, P09 & 10 -> P08, P09 P10
    text = re.sub(r'P(\d+)\s*&\s*(\d+)', r'P\1, P\2',text) #P08 & 10 -> P08,
    text = re.sub(r'P(\d+)\s*,\s*(\d+)\s*,\s*(\d+)', r'P\1, P\2, P\3',text) #P08,09,10 -> P08, P09, P10
    text = re.sub(r'P(\d+)\s*-\s*(\d+)\s*-\s*(\d+)\s*-\s*(\d+)', r'P\1, P\2, P\3 ,P\4',text) #P14-15-16-17 -> P14, P15, P16, P17
    text = re.sub(r'P(\d+)\s*-\s*(\d+)\s*-\s*(\d+)', r'P\1, P\2, P\3', text) #P14-15-16 -> P14, P15, P16
    text = re.sub(r'\b(P\d+)\s*-\s*A\s*-\s*B\s*-\s*C\b', r'\1-A, \1-B, \1-C', text)# P32-A-B-C -> P32-A, P32-B, P32-C
    text = re.sub(r'-(\d+)\s*&\s*-\s*(\d+)', r'P\1, P\2', text) #-32 & - 31
    text= re.sub(r'P(\d+)\s*&\s*-\s*(\d+)', r'P\1, P\2',text) # P44 & -46 -> P44, P46
    text = re.sub(r'P(\d+)\s*x\s*(\d+)\s*&\s*(\d+)', r'P\1 x\2 P\3', text, flags=re.IGNORECASE) # P722 x4 & 726
    text = re.sub(r'P(\d+)\s*x\s*(\d+)\s*,\s*(\d+)', r'P\1 x\2 P\3', text, flags=re.IGNORECASE) # P63 x1,64 x1
    text= re.sub(r'P(\d+)\s*/\s*-\s*(\d+)\s*/\s*-\s*(\d+)\s*/\s*-\s*(\d+)\s*/\s*-\s*(\d+)', r'P\1, P\2, P\3,P\4, P\5', text) # P24/-23/-22/-21/-20 2
    text= re.sub(r'(\d+)\s*/\s*(\d+)\s*/\s*(\d+)\s*/\s*(\d+)\s*/\s*(\d+)\s*/\s*(\d+)', r'P\1, P\2, P\3, P\4, P\5 , P\6', text) # 24/23/22/21/20/21
    text= re.sub(r'P(\d+)\s*x\s*(\d+)\s*/\s*(\d+)\s*x\s*(\d+)\s*/\s*(\d+)\s*x\s*(\d+)\s*/\s*(\d+)\s*x\s*(\d+)\s*/\s*(\d+)\s*x\s*(\d+)\s*/\s*(\d+)\s*x\s*(\d+)\s*/\s*(\d+)\s*x\s*(\d+)', r'P\1 x\2 , P\3 x\4 ,P\5 x\6 , P\7 x\8 ,P\9 x\10 , P\11 x\12 , P\13 x\14', text) # P24 x2 / 27x4 /24 x4 / 22 x2 /21 x2/ 21x2 /20 x2
    text= re.sub(r'P(\d+)\s*/\s*-\s*(\d+)', r'P\1, P\2', text) # P44/-46 -> P44, P46
    text = re.sub(r'(\d+)\s*- (\d+)\s*-\s*&\s*(\d+)\s*-', r'P\1, P\2 ,P\3',text) #11- 12-& 13-
    text = re.sub(r'(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*&\s*(\d+)', r'P\1, P\2 ,P\3 ,P\4', text) #13, 28,30 & 12
    text = re.sub(r'P(\d+),\s*P(\d+)\s*,\s*(\d+)', r'P\1, P\2, P\3',text) #P08, P09 , 10 -> P08, P09, P10
    text = re.sub(r'Placard-(\d+),\s*(\d+)\s*&\s*(\d+)', r'Placard P\1, P\2, P\3', text) #Placard-70, 89 &8
    text = re.sub(r'Assy\s*(\d+)\s*and\s*(\d+)', r'Assy \1, Assy \2', text) # Assy 202 and 302
    text = re.sub(r'"([^"]+)"', r'\1', text) # Supprime les guillemets doubles s'ils entourent un mot ou une phrase
    text = re.sub(r'(\d+)"', r'\1 inch', text)  # Retire les " après les nombres et fait une convertion en unité)
    #text = re.sub(r'(P\d+)(-x)', r'\1 \2', "P300-x1") # P9-x23 -> P9 -x23 j'ai une meilleure version
    text = re.sub(r'(\w)[-=]x', r'\1 x', text) # Séparer les cas comme NUT-x1 ou CABLE=x1 en NUT x1 ou CABLE x1
    text = re.sub(r'-x', r' x', text, flags=re.IGNORECASE) # Remplace -x par espace + x
    text = re.sub(r"-\s*x\s*(\d+)", r" x\1", text)  # P60 -x2 → P60x2
    text = re.sub(r'P(\d+)\s*:\s*x\s*(\d+)', r'P\1 x\2', text, flags=re.IGNORECASE) # #:x -> x
    text = re.sub(r'\s*:\s*x\s*(\d+)', r' x\1',text, flags=re.IGNORECASE) #:x -> x
    text = re.sub(r'\s*,\s*x\s*(\d+)', r' x\1',text, flags=re.IGNORECASE) #:x -> x
    text= re.sub(r'X(\d+)',r'x\1',text,flags=re.IGNORECASE)#X1 -> x1
    text = re.sub(r'\*', ' ', text, flags=re.IGNORECASE) # Remplace * par espace
    text = re.sub(r'\%', ' ', text, flags=re.IGNORECASE) # Remplace % par espace
    text = re.sub(r'\$', ' ', text, flags=re.IGNORECASE) # Remplace $ par espace
    text = re.sub(r'\°', ' ', text, flags=re.IGNORECASE) # Remplace ° par espace
    text = re.sub(r'\#', ' ', text, flags=re.IGNORECASE) # Remplace # par espace
    text = re.sub(r'(P\d+)(-NON)', r'\1 \2',  text, flags=re.IGNORECASE) # P9-x23 -> P9 -x23 j'ai une meilleure version
    text = re.sub(r'(P\d+)(-CUSHION/RUBBER)', r'\1 \2', text, flags=re.IGNORECASE) # P9-x23 -> P9 -x23 j'ai une meilleure version
    text = re.sub(r'(P\d+)(-BOLT)', r'\1 \2', text, flags=re.IGNORECASE) # P9-x23 -> P9 -x23 j'ai une meilleure version
    text = re.sub(r'(P\d+)(-ADHESIVE)', r'\1 \2', text, flags=re.IGNORECASE) # P9-x23 -> P9 -x23 j'ai une meilleure version
    text = re.sub(r'(P\d+)(-GUARD)', r'\1 \2', text, flags=re.IGNORECASE) # P9-x23 -> P9 -x23 j'ai une meilleure version
    text = re.sub(r'(P\d+)(-CABLE)', r'\1 \2', text, flags=re.IGNORECASE) # P9-x23 -> P9 -x23 j'ai une meilleure version
    text = re.sub(r'(P\d+)(-NUT)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-WASHER)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-PAD)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-BLOCKING)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-GRIPHOIST)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-SWIVEL)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(\d+)(CAP)', r'\1 \2', text, flags=re.IGNORECASE) #-34CAP ->-34 CAP
    text = re.sub(r'(\d+)(-LIFTING)', r'\1 \2',text, flags=re.IGNORECASE)#B71040-25-LIFTING
    text = re.sub(r'(P\d+)(,PAWL)', r'\1 \2',text, flags=re.IGNORECASE)#956A8304P21,PAWL
    text = re.sub(r'(P\d+)(-LANYAR)', r'\1 \2',text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-SLEEVE)', r'\1 \2',text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-SHAFT)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-HEX)', r'\1 \2',text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-PLATE)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-DETAIL)', r'\1 \2',text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-TURNIN)', r'\1 \2',text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-REMOVE)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-ALIGNM)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-EXTRAC)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-BOX)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-FOAM)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-PIN)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-CRANK)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-SCR)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-COTTER)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-CURVED)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-CLAMP)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-PULLER)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-COMPRE)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-ORING)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-WIPER)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-CLAMP)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-PULLER)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(-SET)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(Echelle)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(P\d+)(->Rotation)', r'\1 \2', text)
    text = re.sub(r'(P\d+)(->Axial)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(P\d+)\((x\d+)\)', r'\1 (\2)', text, flags=re.IGNORECASE) #P8(x1) ->P8 (x1)
    text = re.sub(r'\b(\d+)\((x\d+)\)', r'\1 (\2)', text, flags=re.IGNORECASE) #8(x1) -> 8 (x1)
    text = re.sub(r'(?<!P)(\b\d+)\s*(\(\s*x\d+\s*\))', r'P\1\2', text, flags=re.IGNORECASE) #6(x2), -6(x2) ,  P6(x2) , -P32(x2) ->P6(x2), -P6(x2) ,  P6(x2) , -P32(x2)
    text = re.sub(r'\b(AF)(P\d+)\b', r'\1 \2', text, flags=re.IGNORECASE) #ROUND ITEM
    text = re.sub(r'\bITEM\s+(?=P\d+)', '', text, flags=re.IGNORECASE)# Supprimer "ITEM" uniquement s'il est suivi d'un P + nombre
    text = re.sub(r'\bITEMS\s+(?=P\d+)', '', text, flags=re.IGNORECASE)# Supprimer "ITEMS" uniquement s'il est suivi d'un P + nombre
    text = re.sub(r'\b(PN:)(\S)', r'\1 \2', text, flags=re.IGNORECASE) #PN:98F07203020208 -> PN: 98F07203020208
    text = re.sub(r'\b(P/N)(-\S)', r'\1 \2', text, flags=re.IGNORECASE) #P/N-98F07203020208 -> P/N -98F07203020208
    text = re.sub(r'>>', r' to ',text) #8C2581G02>>8C2581G03 -> 8C2581G02 to 8C2581G03
    text = re.sub(r'(?<![\w/-])([A-Z]+)-(\d+)', r'\1 -\2',text )#"VALVE-6 -> "VALVE -6
    text = re.sub(r'P(\d+)\s*rep\s*([A-Z]+)',r'P\1-\2',text) #P20 rep A -> P20-A
    text = re.sub(r'-P(\d+)',r'P\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()



In [8]:
nettoyer_description(raw_text)

'1. PLUNGER P60 x2 2. SEAL RUBBER P130 x1. 3. COUPLER P430 x2 4. STRAPSP360 x1. 5. DUCT FLEXIBLE P70 x1 6. KIT CLAMP P20 x1. 7. SCREW SOCKET HEAD CAP P110 x2 8. FLAT WASHER P210 x2 9. HOSE (25 FT) P50 x1. 10. COVER DRAIN P20 x1. P11.P044 PLUMB LINE P/N: 98L21508593.044 x2. 11. P130 12. P12 WEB SLING x1. P26 BOLT x1 13. 1. AIR CHUCK P6 x1. 2. HOSE ASSY P39 x1. 3. NEEDLE VALVE P4 x1. 14. A LOAD ASSY -58 E M83723/60-18AR x1. -2 ASSY 7101SYCQE 89 & 91 x1. 15. 0-16 BAR GAUGE x1. 16. 1. FLOURESCENT TUBE 48 x4 17. 1. CASTERING WHEEL P21 x2 2. CASTERING WHEEL STOP FIX P25 x2 18. 1. WING THUMB NUT x1. 2. DIAL INDICATOR CABLE x1. 3. EXTENSION CABLE x1. séparer NUT de x1 phrase 300 19. 1. WING THUMB NUT x1. 2. DIAL INDICATOR CABLE x1. 3. EXTENSION CABLE x1. 4. P09 x1 20. 1. TENSIONER PAD P22 x1. 2. CABLE ASSEMBLY P26 x1. 21. P532 -NON 22. P23 HAMMER DRIVE x4. P25 MARKER x1. 23. Below items to replace: P6 /8 x1 P10 x4 P11 x8 P9 x4 24. 1. Streamer to be changed 25. text = 1. MS3181-10RAL-6 x1 missi

## 4. Données d'entraînement

In [10]:

TRAIN_DATA = [
    (
        "Ajout de la pièce P12GH-20-28-7 cassé avec une quantité x2",
        {"entities": [(0, 5, "ACTION"), (18, 31, "PART"), (32, 37, "OBS"), (56, 58, "QTY")]}
    ),
    (
        "La pièce p12x3 a été removed car manquant",
        {"entities": [(9, 14, "PART"), (21, 28, "ACTION"), (33, 41, "OBS")]}
    ),
    (
        "Fourniture et changement du joint REP021 fissuré x1",
        {"entities": [(14, 24, "ACTION"), (34, 40, "PART"), (41, 48, "OBS"), (49, 51, "QTY")]}
    ),
    (
        "Suppression de la pièce P99X-44-90 manquant",
        {"entities": [(0, 11, "ACTION"), (24, 34, "PART"), (35, 43, "OBS")]}
    ),
    (
        "S711V1001125 manque Item 53",
        {"entities": [
            (13, 19, "OBS"),        # Manque
            (20, 27, "PART")       # Item 53
        ]}
    ),
    (
    " S711V1001231 Configuration label Item 004 unvisible: to replace",
    {"entities": [
        (34, 42, "PART"),       # Item 004
        (43, 52, "OBS"),        # unvisible
        (57, 64, "ACTION")     # replace

    ]}
    ),
    (
    " S711V1001232 Stop missing (x2): welding +painting rework",
    {"entities": [

        (19, 26, "OBS"),        # missing
        (28, 30, "QTY")       # x2
    ]}
    ),
    (
        "item 007 missing to be provided",
        {"entities": [(0,8, "PART"),]}
    ),
    (
        "S711V1001804 Hook bended: to be replaced",
        {"entities": [(13,24, "DESCRIPTION"),]}
    ),
    (
        "cylindre 601-001 bague de guidage 602-001 changement de tout les joints de l'ensemble  vérin .",
        {"entities": [(9,16, "PART"),(34,41, "PART"),]}
    ),
    (
        "-fourniture et changement du joint racleur (P12GH-20-28-7) et joint à lèvre (P38-20-28-7) de la pompe à main .",
        {"entities": [(29, 42, "DESCRIPTION"),(44,57, "PART"),(62, 75, "DESCRIPTION"),(77,88, "PART"),]}
    ),
    (
        "Sangle E240-8T Type MEGA Longueur 1300 mm. en fibre polyester.",
        {"entities": [(0,6, "DESCRIPTION"),(7,14, "PART"),(15,24, "DESCRIPTION") ,]}
    ),
    (
        "1 crochet ref:S711V1001804 1 sangle ref:S711V1001122 1 plaque configuration ref: S711V1001231.",
        {"entities": [(0, 1, "QTY"),(2,9, "DESCRIPTION"),(14,26, "PART"),(27, 28, "QTY"),(29,35, "DESCRIPTION"),(40,52, "PART"),(53, 54, "QTY"),(55,61, "DESCRIPTION"),(62, 75, "ACTION"),(81,93, "PART") ,]}
    ),
    (
        "-changement du joint racleur et du joint torique de la pompe à main .",
        {"entities": [(0, 11, "ACTION"),(15,28, "DESCRIPTION"),(35,48, "DESCRIPTION"),]}
    ),
    (
        "-changement du joint racleur et du joint torique de la pompe à main .",
        {"entities": [(0, 11, "ACTION"),(15,28, "DESCRIPTION"),(35,48, "DESCRIPTION"),]}
    ),
    (
        "1. PLUNGER P60 x2 2. SEAL RUBBER P130 = x1. 3. COUPLER P430 x2 4. STRAPSP360 = x1. 5. DUCT FLEXIBLE P70 = x1 6. KIT CLAMP P20 = x1. 7. SCREW SOCKET HEAD CAP P110 x2 8. FLAT WASHER P210 x2 9. HOSE (25 FT) P50 = x1. 10. COVER DRAIN P20 = x1.",
        {"entities": [(3,10,"DESCRIPTION"),(11,14, "PART"),(15,17, "QTY"),(21,32,"DESCRIPTION"),(33,37, "PART"),(40,42, "QTY"),(47,54,"DESCRIPTION"),(55,59, "PART"),(60,62, "QTY"),(66,76,"PART"),(79,81, "QTY"),(86,99, "DESCRIPTION"),(100,103, "PART"),(106,108, "QTY"),(112,121,"DESCRIPTION"),(122,125, "PART"),(128,130, "QTY"),
                      (135,156,"DESCRIPTION"),(157,161, "PART"),(162,164, "QTY"),(168,179,"DESCRIPTION"),(180,184, "PART"),(185,187, "QTY"),(191,202,"DESCRIPTION"),(204,207, "PART"),(210,212, "QTY"),(218,229,"DESCRIPTION"),(230,233, "PART"),(236,238, "QTY"),]}
     ),
    (
         "ITEM P044 PLUMB LINE P/N: 98L21508593.044 x2.",
         {"entities": [(5, 9, "PART"),(10,20, "DESCRIPTION"),(42,44, "QTY"),]}
     ),
    (
         "ROUND STEEL BAR P246 = x1.",
         {"entities": [(0,15, "DESCRIPTION"),(16, 20, "PART"),(23,25, "QTY"),]}
     ),
    (
         "THREADED PIN P394 x2",
         {"entities": [(0,12, "DESCRIPTION"),(13, 17, "PART"),(18,20, "QTY"),]}
     ),
    (
         "ITEM P202 PROTECT IV - x1.",
         {"entities": [(5, 9, "PART"),(10,20, "DESCRIPTION"),(23,25, "QTY"),]}
     ),
    (
         "P12 WEB SLING x1. P26 BOLT = x1",
         {"entities": [(0, 3, "PART"),(4,13, "DESCRIPTION"),(14,16, "QTY"),(18,21, "PART"),(22,26, "DESCRIPTION"),(29,31, "QTY"),]}
     ),
    (
         "1. BOLT P31 x6 2. CUSHION P8 x3",
         {"entities": [(3,7, "DESCRIPTION"),(8,11, "PART"),(12,14, "QTY"),(18,25, "DESCRIPTION"),(26,28, "PART"),(29,31, "QTY"),]}
     ),
    (
         "ITEM P15 SREW x2.",
         {"entities": [(5,8, "PART"),(9,13, "DESCRIPTION"),(14,16, "QTY"),]}
     ),
    (
         "1. MASTER LINK P134 = x1. 2. WASHER P7 x2 ",
         {"entities": [(3,14, "DESCRIPTION"),(15,19, "PART"),(22,24, "QTY"),(29,35, "DESCRIPTION"),(36,38, "PART"),(39,41, "QTY"),]}
     ),
    (
         "ITEM P9 RONDELLE WASHER = x1 ITEM P16 VIS CHC SCREW x2 ITEM P5 EXTRACTOR NUT x2 ITEM P15 SCREW x2",
         {"entities": [(5,7, "PART"),(8,23, "DESCRIPTION"),(26,28, "QTY"),(34,37, "PART"),(38,51, "DESCRIPTION"),(52,54, "QTY"),(60,62, "PART"),(63,76, "DESCRIPTION"),(77,79, "QTY"),(85,88, "PART"),(89,94, "DESCRIPTION"),(95,97, "QTY"),]}
     ),
    (
         "ITEM P058 - PROTECT - x1. ITEM P060 - PROTECT 2 - x2.",
         {"entities": [(5,9, "PART"),(12,19, "DESCRIPTION"),(22,24, "QTY"),(31,35, "PART"),(38,45, "DESCRIPTION"),(50,52, "QTY"),]}
     ),
    (
         "1. NYLON PROTECTION = P008 - x2 2. NYLON PROTECTION = P009 - x1",
         {"entities": [(3,19, "DESCRIPTION"),(22,26, "PART"),(29,31, "QTY"),(35,51, "DESCRIPTION"),(54,58, "PART"),(61,63, "QTY"),]}
     ),
    (
         "ITEM P238 PROTECTION x4.",
         {"entities": [(5,9, "PART"),(10,20, "DESCRIPTION"),(21,23, "QTY"),]}
     ),
    (
         "ITEM P26 SCREW x12.",
         {"entities": [(5,8, "PART"),(9,14, "DESCRIPTION"),(15,18, "QTY"),]}
     ),
    (
         "1. HEX HEAD BOLT = x1. 2. NUT x8 3. WASHER x8",
         {"entities": [(3,16, "DESCRIPTION"),(19,21, "QTY"),(26,29, "DESCRIPTION"),(30,32, "QTY"),(36,42, "DESCRIPTION"),(43,45, "QTY"),]}
     ),
    (
         "ITEM P634 SCREW - x3. ITEM P640 WASHER - x3. ITEM P642 HEX. NUT - x3. ITEM P580 RING SREW - x1. ITEM # HAND WHEEL - x1 ITEM # CASTER WHEELS - x4",
         {"entities": [(5,9, "PART"),(10,15, "DESCRIPTION"),(18,20, "QTY"),(27,31, "PART"),(32,38, "DESCRIPTION"),(41,43, "QTY"),(50,54, "PART"),(55,63, "DESCRIPTION"),(66,68, "QTY"),(75,79, "PART"),(80,89, "DESCRIPTION"),(92,94, "QTY"),(103,113, "DESCRIPTION"),(116,118, "QTY"),(126,139, "DESCRIPTION"),(142,144, "QTY"),]}
     ),
    (
         "SIDE REFLECTOR x2. HANDLE AS PER SAMPLE x1. FLOW METER x1.",
         {"entities": [(0,14, "DESCRIPTION"),(15,17, "QTY"),(19,39, "DESCRIPTION"),(40,42, "QTY"),(44,54, "DESCRIPTION"),(55,57, "QTY"),]}
     ),
    (
         "115 BALL LOCK PIN x1. FLAT HEAD SOCKET SCREW x1.",
         {"entities": [(4,17, "DESCRIPTION"),(18,20, "QTY"),(22,44, "DESCRIPTION"),(45,47, "QTY"),]}
     ),
    (
         "1. CLEVIS 305-1 = x1. 2. STRUT BODY 305-2 = x1. 3. SHOULDER BOLT 305-3 = x1.",
         {"entities": [(3,9, "DESCRIPTION"),(10,15, "PART"),(18,20, "QTY"),(25,35, "DESCRIPTION"),(36,41, "PART"),(44,46, "QTY"),(51,64, "DESCRIPTION"),(65,70, "PART"),(73,75, "QTY"),]}
     ),
    (
         "WROUGHT ALU ALLOY FERRULES",
         {"entities": [(0,26, "DESCRIPTION") ,]}
     ),
    (
         "EYELETS FOR CANVAS",
         {"entities": [(0,18, "DESCRIPTION") ,]}
     ),
    (
         "RING SPLIT IN STEELS , NUT , PIN , E-CLIPS RING",
         {"entities": [(0,20, "DESCRIPTION"),(23,26, "DESCRIPTION"),(29,32, "DESCRIPTION"),(35,47, "DESCRIPTION") ,]}
     ),
    (
         "ASSY -15 -31 ",
         {"entities": [(0,12, "PART") ,]}
     ),
    (
         "K71001-22",
         {"entities": [(0,9, "PART") ,]}
     ),
    (
         "1. PRESSURE GAUGE P67 ON ASSY-10 = x1. 2. BALL VALVE P53 ON ASSY-7 = x1. 3. VENT ASSY P12 = x1. 4. COUPLER P57 ON ASSY-6 x4",
         {"entities": [(3,17, "DESCRIPTION"),(18,21, "PART"),(25,32, "PART"),(35,37, "QTY"),(42,52, "DESCRIPTION"),(53,56, "PART"),(60,66, "PART"),(69,71, "QTY"),(76,85,"DESCRIPTION"),(86,89, "PART"),(92,94, "QTY"),
                       (99,106,"DESCRIPTION"),(107,110, "PART"),(114,120, "PART"),(121,123, "QTY"),]}
     ),
    (
         "-21 SHOULDER SCREW x2. -20 SHOULDER SREW x2. -41 MALE CONNECTOR x1. -28 SEAL x1. -44 TYGON TUBING x1. -14 LOWER PAD x1.",
         {"entities": [(0,3, "PART"),(4,18, "DESCRIPTION"),(19,21, "QTY"),(23,26, "PART"),(27,40, "DESCRIPTION"),(41,43, "QTY"),(45,48, "PART"),(49,63, "DESCRIPTION"),(64,66, "QTY"),(68,71, "PART"),(72,76, "DESCRIPTION"),(77,79, "QTY"),(81,84, "PART"),(85,97, "DESCRIPTION"),
                       (98,100, "QTY"),(102,105, "PART"),(106,115, "DESCRIPTION"),(116,118, "QTY"),]}
     ),
    (
         "P34 CAP - x1. P9 /-23 CONNECTOR - x1. P4 NEEDLE VALVE x1. P8 NIPPLE x2. P25 RUBBER BOOT x1. P29 GAUGE x1. P3 CHECK VALVE x1. P28 TEE x1. P11 REDUCER x1. P24 SHRINK TUBE 1PC",
         {"entities": [(0,3, "PART"),(4,7, "DESCRIPTION"),(10,12, "QTY"),(14,16, "PART"),(22,31, "DESCRIPTION"),(34,36, "QTY"),(38,40, "PART"),(41,53, "DESCRIPTION"),(54,56, "QTY"),(58,60, "PART"),(61,67, "DESCRIPTION"),(68,70, "QTY"),(72,75, "PART"),(76,87, "DESCRIPTION"),
                       (88,90, "QTY"),(92,95, "PART"),(96,101, "DESCRIPTION"),(102,104, "QTY"),(106,108, "PART"),(109,120, "DESCRIPTION"),(121,123, "QTY"),(125,128, "PART"),(129,132, "DESCRIPTION"),(133,135, "QTY"),(137,140, "PART"),(141,148, "DESCRIPTION"),(149,151, "QTY"),(153,156, "PART"),(157,168, "DESCRIPTION"),(169,172, "QTY") ,]}
     ),
    (
         "1. FOOT HOIST SUCTION P5 ON ASSY A1/61/815 x3",
         {"entities": [(3,21, "DESCRIPTION"),(22,24, "PART"),(25,42, "DESCRIPTION"),(43,45, "QTY") ,]}
     ),
    (
         "1. BALL LOCK PIN P502 x4ON ASSY 042 2. CIRCLIPS P514 x2 ON ASSY 042 3. WASHER P522 x2 ON ASSY 042 4. BALL LOCK PIN P518 = x1. ON ASSY 114",
         {"entities": [(3,16, "DESCRIPTION"),(17,21, "PART"),(27,35, "DESCRIPTION"),(39,47, "DESCRIPTION"),(48,52, "PART"),(53,55, "QTY"),(56,67, "DESCRIPTION"),(71,77, "DESCRIPTION"),(78,82, "PART"),(83,85, "QTY"),(86,97, "DESCRIPTION"),(101,114, "DESCRIPTION"),(115,119, "PART"),
                     (122,124, "QTY"),(126,137, "DESCRIPTION") ,]}
     ),
    (
         "1. NAS1338C5C25D - PIN QUICK RLSE 1/2 - x1. 2. NAS1334C5C20D - PIN QUICK RLSE 1/4 - x2. 3. P60 PIN RETAINER - x2. 4. P28 PLAIN WASHER - x4. 5. NAS6606-8 BOLT - x4.",
         {"entities": [(3,16, "PART"),(19,28, "DESCRIPTION"),(40,42, "QTY"),(47,60, "PART"),(63,72, "DESCRIPTION"),(84,86, "QTY"),(91,94, "PART"),(95,107, "DESCRIPTION"),(110,112, "QTY"),(117,120, "PART"),(121,133, "DESCRIPTION"),(136,138, "QTY"),(143,152, "PART"),(153,157, "DESCRIPTION"),(160,162, "QTY") ,]}
     ),
    (
         "KEY x1.",
         {"entities": [(0,3, "DESCRIPTION"),(4,6, "QTY") ,]}
     ),
    (
         "98D71203501042 ITEM P3 BLS12TA16S x1.",
         {"entities": [(0,14, "PART"),(20,22, "PART"),(23,33, "DESCRIPTION"),(34,36, "QTY") ,]}
     ),
    (
         "1. ASSY -33 CLEVIS x1. 2. P40 SCREW x1. 3. P44 CABLE ASSY x1. 4. ITEM P43 BALL LOCK PIN x2. 5. ITEM # LOAD CELL x1",
         {"entities": [(3,11, "PART"),(12,18, "DESCRIPTION"),(19,21, "QTY"),(26,29, "PART"),(30,35, "DESCRIPTION"),(36,38, "QTY"),(43,46, "PART"),(47,57, "DESCRIPTION"),(58,60, "QTY"),(70,73, "PART"),(74,87, "DESCRIPTION"),(88,90, "QTY"),(102,111, "DESCRIPTION"),(112,114, "QTY") ,]}
     ),
  (
       "1. U-BOLT P26 = x1. 2. U-BOLT P24 x3 3. HEX NUT 3/8-16 x8 4. PULL RING ON P27 = x1. 5. P30 DETENT PIN = x1.",
       {"entities": [(3,9, "DESCRIPTION"),(10,13, "PART"),(16,18, "QTY"),(23,29, "DESCRIPTION"),(30,33, "PART"),(34,36, "QTY"),(40,47, "DESCRIPTION"),(51,54, "PART"),(55,57, "QTY"),(61,73, "DESCRIPTION"),(74,77, "PART"),(80,82, "QTY"),(87,90, "PART"),(91,101, "DESCRIPTION"),(104,106, "QTY") ,]}
   ),
  (
       "ITEM P710 - H SCREW M6X12",
       {"entities": [(5,9, "PART"),(12,25, "DESCRIPTION") ,]}
   ),
  (
       "1. STORAGE BOX ASSY = x1. 2. HOIST RING P13 x2",
       {"entities": [(3,19, "DESCRIPTION"),(22,24, "QTY"),(29,39, "DESCRIPTION"),(40,43, "PART"),(44,46, "QTY") ,]}
   ),
  (
       "-401 ASSY Washer x4 -401 ASSY Retaining Ring x3",
       {"entities": [(0,16, "DESCRIPTION"),(17,19, "QTY"),(20,44, "DESCRIPTION"),(45,47, "QTY") ,]}
   ),
  (
       "1. USB CABLE 1. 8 M = x1. 2. EQUOSTAT 3 TEST BLOCK CALIBRATED = x1.",
       {"entities": [(3,19, "DESCRIPTION"),(22,24, "QTY"),(29,61, "DESCRIPTION"),(64,66, "QTY") ,]}
   ),
  (
       "ITEM P54 PIN LOC x2. ITEM P84 NUT THIN x1. ITEM P85 WASHER SPRING S/COIL x1. ITEM P93 TAG ATTACHMENT x2.",
       {"entities": [(5,8, "PART"),(9,16, "DESCRIPTION"),(17,19, "QTY"),(26,29, "PART"),(30,38, "DESCRIPTION"),(39,41, "QTY"),(48,51, "PART"),(52,72, "DESCRIPTION"),(73,75, "QTY"),(82,85, "PART"),(86,100, "DESCRIPTION"),(101,103, "QTY") ,]}
   ),
  (
       "A LOAD ASSY -58 E M83723/60-18AR x1. -2 ASSY 7101SYCQE 89 & 91 x1.",
       {"entities": [(0,15, "DESCRIPTION"),(18,32, "PART"),(33,35, "QTY"),(37,62, "DESCRIPTION"),(63,65, "QTY") ,]}
   ),
  (
       "1. BALL LOCK PIN = x1. (S) 1. BALL LOCK PIN = x1. (L)",
       {"entities": [(3,16, "DESCRIPTION"),(19,21, "QTY"),(30,43, "DESCRIPTION"),(46,48, "QTY") ,]}
   ),
  (
       "1. Hook (M6 / M10) = 1 Each 2. Cone tip (M6 / M10) = 1 Each 3. Flat head (M6 / 10) = 1 Each 4. Notched head (M6 / FM10) = 1 Each 5. Universal VAC charger with EC-, UK- and US wall plug connector = x1. Each 6. USB-cable (FMI-946) = x1.",
       {"entities": [(3,18, "DESCRIPTION"),(31,50, "DESCRIPTION"),(63,82, "DESCRIPTION"),(95,119, "DESCRIPTION"),(132,162, "DESCRIPTION"),(209,228, "DESCRIPTION"),(231,233, "QTY") ,]}
   ),
  (
       "THREADED COUPLING CONNECTOR , BOLT HEX HD",
       {"entities": [(0,27, "DESCRIPTION"),(30,41, "DESCRIPTION") ,]}
   ),
  (
       "ASSY 98F32201000042 NOT INCLUDED TO 98F32201005000 BUT DAMAGE BALL LOCK PIN x1.",
       {"entities": [(0,75, "DESCRIPTION"),(76,78, "QTY") ,]}
   ),
  (
       "ASSY -69 SPLIT BUSHING x1. ASSY -33 FUSE x2.",
       {"entities": [(0,8, "PART"),(9,22, "DESCRIPTION"),(23,25, "QTY"),(27,35, "PART"),(36,40, "DESCRIPTION"),(41,43, "QTY") ,]}
   ),
  (
       "ITEMS P111 - 16 EXTENSION POLE = x1",
       {"entities": [(6,10, "PART"),(16,30, "DESCRIPTION"),(33,35, "QTY") ,]}
   ),
  (
       "ITEMS P111 - 16 EXTENSION POLE = x1",
       {"entities": [(6,10, "PART"),(16,30, "DESCRIPTION"),(33,35, "QTY") ,]}
   ),
  (
       "ASSY -101 RUBBER PAD x2. ASSY -8 RUBBER PAD x1. ASSY -8 CLAMP ASSY x1. ASSY -8 KEY INSERT x1. ASSY -9 RUBBER PAD x1. ASSY -9 CLAMP ASSY. x1.",
       {"entities": [(0,9, "PART"),(10,20, "DESCRIPTION"),(21,23, "QTY"),(25,32, "PART"),(33,43, "DESCRIPTION"),(44,46, "QTY"),(48,55, "PART"),(56,66, "DESCRIPTION"),(67,69, "QTY"),(71,78, "PART"),(79,89, "DESCRIPTION"),(90,92, "QTY"),
                     (94,101, "PART"),(102,112, "DESCRIPTION"),(113,115, "QTY"),(117,124, "PART"),(125,135, "DESCRIPTION"),(137,139, "QTY"),]}
   ),
  (
       "-26 CABLE ASSY x1 -27 WARNING STREAMER x1 -34 CAP x1 -6 AIR CHUCK x1",
       {"entities": [(0,3, "PART"),(4,14, "DESCRIPTION"),(15,17, "QTY"),(18,21, "PART"),(22,38, "DESCRIPTION"),(39,41, "QTY"),(42,45, "PART"),(46,49, "DESCRIPTION"),(50,52, "QTY"),(53,55, "PART"),(56,65, "DESCRIPTION"),(66,68, "QTY") ,]}
   ),
  (
       "ASSY 118 BALL LOCK PIN x2. ASSY 114 MS17984C1226 BALL LOCK PIN x1. ASSY 116 BALL LOCK PIN x1. ASSY 224 BALL LOCK PIN x1. ASSY 222 CIRCLIP x1. ASSY 222 WASHER x1.",
       {"entities": [(0,8, "PART"),(9,22, "DESCRIPTION"),(23,25, "QTY"),(27,48, "PART"),(49,62, "DESCRIPTION"),(63,65, "QTY"),(67,75, "PART"),(76,89, "DESCRIPTION"),(90,92, "QTY"),(94,102, "PART"),(103,116, "DESCRIPTION"),(117,119, "QTY"),
                     (121,129, "PART"),(130,137, "DESCRIPTION"),(138,140, "QTY"),(142,150, "PART"),(151,157, "DESCRIPTION"),(158,160, "QTY") ,]}
   ),
  (
       "1. PLUMB LINE ASSY 044 x2",
       {"entities": [(3,13, "DESCRIPTION"),(14,22, "PART"),(23,25, "QTY") ,]}
   ),
  (
       "1. Hose 2m/6’7”, 8V1 - x1. 2. INLET FITTING - x1.",
       {"entities": [(3,20, "DESCRIPTION"),(23,25, "QTY"),(30,43, "DESCRIPTION"),(46,48, "QTY") ,]}
   ),
  (
       "1,switch button cover 2. CHARGER",
       {"entities": [(0,21, "DESCRIPTION"),(25,32, "DESCRIPTION") ,]}
   ),
  (
       "1, CASTOR PLATE P5 x3 2, SPRING GUIDE(LOWER) P6 x3 3, SPRING GUIDE(UPPER) P7 x3 4, PLAIN WASHER P38 x3 5, HEX HEAD SCREW P45 x3 6, PLAIN WASHER P49-x3 7, COMPRESSION SPRING x3 8, SPRING WASHER x3 9, SPRING LOCK WASHER PIN x9 10, LASHING",
       {"entities": [(3,15, "DESCRIPTION"),(16,18, "PART"),(19,21, "QTY"),(25,44, "DESCRIPTION"),(45,47, "PART"),(48,50, "QTY"),(54,73, "DESCRIPTION"),(74,76, "PART"),(77,79, "QTY"),(83,95, "DESCRIPTION"),(96,99, "PART"),(100,102, "QTY"),
                     (106,120, "DESCRIPTION"),(121,124, "PART"),(125,127, "QTY"),(131,143, "DESCRIPTION"),(154,172, "DESCRIPTION"),(173,175, "QTY"),(179,192, "DESCRIPTION"),(193,195, "QTY"),(199,221, "DESCRIPTION"),(222,224, "QTY"),(229,236, "DESCRIPTION"),]}
   ),
  (
       "VALVE-CHECK , VALVE-NEEDLE , PRESSURE GAUGE 400 BAR",
       {"entities": [(0,11, "DESCRIPTION"),(29,51, "DESCRIPTION") ,]}
   ),
  (
       "2 shouldered screws 90269A141",
       {"entities": [(2,19, "DESCRIPTION"),(20,29, "PART") ,]}
   ),
  (
       "BOLT, HEX, HEAD SPECIAL",
       {"entities": [(0,23, "DESCRIPTION") ,]}
   ),
  (
       "HEX HD SCREW , HAND CRUSH LABEL , PLUG, MALE THREAD",
       {"entities": [(0,12, "DESCRIPTION"),(15,31, "DESCRIPTION"),(34,51, "DESCRIPTION") ,]}
   ),
  (
       "PN:N-20009-16-B , PN:N-2403-04-B , P351-2100_14GSEV01_C_DET-302-1 , YL500",
       {"entities": [(3,15, "PART"),(21,32, "PART"),(35,65, "PART"),(68,73, "PART") ,]}
   ),
  (
       "P119 TAG x1. ITEM P110 TURNBUCKLE x3. ITEM P101-2 PAD x1. P117 NUT x8 , P533-VHK",
       {"entities": [(0,4, "PART"),(5,8, "DESCRIPTION"),(9,11, "QTY"),(18,22, "PART"),(23,33, "DESCRIPTION"),(34,36, "QTY"),(43,49, "PART"),(50,53, "DESCRIPTION"),(54,56, "QTY"),(58,62, "PART"),(63,66, "DESCRIPTION"),(67,69, "QTY"),(72,80, "PART") ,]}
   ),
  (
       "NUT (SELF LOCKING) , PIN-QUICK RELEASE , HANDLE , ADAPTER",
       {"entities": [(0,3, "DESCRIPTION"),(21,38, "DESCRIPTION"),(41,47, "DESCRIPTION"),(50,57, "DESCRIPTION") ,]}
   ),
  (
       "ASSY 040 RACCORD x1.",
       {"entities": [(0,8, "PART"),(9,16, "DESCRIPTION"),(17,19, "QTY") ,]}
   ),
  (
       "SCREW PRE-MOD , SCREW POST-MOD , SHACKLE , Additional spares ,  HOOK , 1 1/16 inch , Torque Mulitplier",
       {"entities": [(0,13, "DESCRIPTION"),(16,30, "DESCRIPTION"),(33,40, "DESCRIPTION"),(43,60, "DESCRIPTION"),(64,68, "DESCRIPTION"),(71,82, "DESCRIPTION"),(85,102, "DESCRIPTION") ,]}
   ),

  (
       "1.( PVC SUCTION HOSE ) , ( STATIC ADAPTOR ) , Failed calibration - Out of specification , PIP-PIN",
       {"entities": [(4,20, "DESCRIPTION"),(27,41, "DESCRIPTION"),(46,87, "DESCRIPTION"),(90,97, "DESCRIPTION")  ,]}
   ),
  (
       "MS3180-10RA  ,  MS3476A10-6S , M85049/52-1-10A , ASSY -3 M/W MARKER x1 , ASSY 2000 SCREW",
       {"entities": [(0,11, "PART"),(16,28, "PART"),(31,46, "PART"),(49,60, "PART"),(61,67, "DESCRIPTION"),(73,88, "DESCRIPTION") ,]}
   ),
  (
       "1. LONG BOLT (P4) - x1",
       {"entities": [(3,12, "DESCRIPTION"),(14,16, "PART"),(20,22, "QTY") ,]}
   ),
  (
       "ASSY 53 P48 BRIDLE - x1",
       {"entities": [(0,11, "PART"),(12,18, "DESCRIPTION"),(21,23, "QTY") ,]}
   ),
  (
       "TORQUE WRENCHES - x2 (Provided by the customer)",
       {"entities": [(0,15, "DESCRIPTION"),(18,20, "QTY"),(22,46, "DESCRIPTION"),]}
   ),

  (
       "ASSY 002 P30 BALL LOCK PIN x1 DILLON DYNAMOMETER SN:E06258 BEARING",
       {"entities": [(0,12, "PART"),(13,26, "DESCRIPTION"),(27,29, "QTY"),(30,66, "DESCRIPTION"),]}
   ),

  (
       "1. ASSY 3 INTERCONNECT CABLE (x1) 2. ASSY 3 P20 PROTECTIVE CAP (x1) 3. ASSY 2 LOAD CELL - CONNECTOR (x1).",
       {"entities": [(3,9, "PART"),(10,28, "DESCRIPTION"),(30,32, "QTY"),(37,47, "PART"),(48,62, "DESCRIPTION"),(64,66, "QTY"),(71,77, "PART"),(78,99, "DESCRIPTION"),(101,103, "QTY"),]}
   ),
  (
       "9400A-146 CLINOMETER READOUT x1. 9300A-112 LOADCELL READOUT x1. FUSE x3.",
       {"entities": [(0,9, "PART"),(10,28, "DESCRIPTION"),(29,31, "QTY"),(33,42, "PART"),(43,59, "DESCRIPTION"),(60,62, "QTY"),(64,68, "DESCRIPTION"),(69,71, "QTY"),]}
   ),
  (
       "02111002-000 ,  P/N:25-1009AW-02L-15",
       {"entities": [(0,12, "PART"),(16,36, "PART"),]}
   ),
  (
       "Change of main circuit board and memory cards needed, change of housing to other control unit. S/N 1792 -> 1474",
       {"entities": [(0,111, "DESCRIPTION"),]}
   ),
  (
       "LH HOLD OPEN ASSY x1 RH HOLD OPEN ASSY x1 -12 LOWER PAD x2 -11 FLAT CSK HEAD CAP SCREW x10 -13 LH UPPER PAD x2 -14 RH UPPER PAD x2 -17 LANYARD x2",
       {"entities": [(0,17, "DESCRIPTION"),(18,20, "QTY"),(21,38, "DESCRIPTION"),(39,41, "QTY"),(42,45, "PART"),(46,55, "DESCRIPTION"),(56,58, "QTY"),(59,62, "PART"),
                     (63,86, "DESCRIPTION"),(87,90, "QTY"),(91,94, "PART"),(95,107, "DESCRIPTION"),(108,110, "QTY"),(111,114, "PART"),(115,127, "DESCRIPTION"),(128,130, "QTY"),(131,134, "PART"),(135,142, "DESCRIPTION"),(143,145, "QTY"),]}
   ),
  (
       "-86 SPRING PIN x4 -62 HOIST PIN x2 -12 COLLAR x2 -71 STRAP ASSY x2",
       {"entities": [(0,3, "PART"),(4,14, "DESCRIPTION"),(15,17, "QTY"),(18,21, "PART"),(22,31, "DESCRIPTION"),(32,34, "QTY"),(35,38, "PART"),(39,45, "DESCRIPTION"),(46,48, "QTY"),(49,52, "PART"),(53,63, "DESCRIPTION"),(64,66, "QTY"),]}
   ),
  (
       "120 FLAT HEAD SOCKET SCREW x2 121 FLAT HEAD SOCKET SCREW x4 114 COMPRESSION SPRING x1 111 EXTENSION POLE x1 116 ALUM RND TUBING x1 113 DOWEL PIN x1",
       {"entities": [(0,3, "PART"),(4,26, "DESCRIPTION"),(27,29, "QTY"),(30,33, "PART"),(34,56, "DESCRIPTION"),(57,59, "QTY"),(60,63, "PART"),(64,82, "DESCRIPTION"),(83,85, "QTY"),(86,89, "PART"),(90,104, "DESCRIPTION"),(105,107, "QTY"),
                     (108,111, "PART"),(112,127, "DESCRIPTION"),(128,130, "QTY"),(131,134, "PART"),(135,144, "DESCRIPTION"),(145,147, "QTY"),]}
   ),
  (
       "Failed load test 1. P33 PIN - x1 2. P55 SHORT PIN - x1",
       {"entities": [(0,16, "DESCRIPTION"),(20,23, "PART"),(24,27, "DESCRIPTION"),(30,32, "QTY"),(36,39, "PART"),(40,49, "DESCRIPTION"),(52,54, "QTY"),]}
   ),
  (
       "ASSY 050 P525 WASHER",
       {"entities": [(0,13, "PART"),(14,20, "DESCRIPTION"),]}
   ),
  (
       "ASSY 046 P12 (TUBE) - x1 ASSY 046 P9 (MALE UNION) - x2 ASSY 042 P6 (TREADED ROD) - x1 ASSY 042 P32 (GOUPILLE PIN) - x1 ASSY 042 P33 (GOUPILLE PIN) - x1",
       {"entities": [(0,12, "PART"),(14,18, "DESCRIPTION"),(22,24, "QTY"),(25,36, "PART"),(38,48, "DESCRIPTION"),(52,54, "QTY"),(55,66, "PART"),(68,79, "DESCRIPTION"),(83,85, "QTY"),(86,98, "PART"),(100,112, "DESCRIPTION"),(116,118, "QTY"),
                     (119,131, "PART"),(133,145, "DESCRIPTION"),(149,151, "QTY"),]}
   ),
  (
       "ASSY-83 ARM ASSY MISSING x1 BALL LOCK PIN x1 LANYARD WITH SLEEVE x1 SPHIRICAL BEARING x1",
       {"entities": [(0,7, "PART"),(8,24, "DESCRIPTION"),(25,27, "QTY"),(28,41, "DESCRIPTION"),(42,44, "QTY"),(45,64, "DESCRIPTION"),(65,67, "QTY"),(68,85, "DESCRIPTION"),(86,88, "QTY"),]}
   ),
  (
       "Pressure Gauge , DEMI-SEGMENT HALF-SEGMENT",
       {"entities": [(0,14, "DESCRIPTION"),(17,42, "DESCRIPTION"),]}
   ),
  (
       "018 SHACKLE x1 007 EYEBOLT x1 020 HEXAGON NUT x1 019 TURNBUCKLE x1 021 HEXAGON NUT x1 009 FORK BOLT x1",
       {"entities": [(0,3, "PART"),(4,11, "DESCRIPTION"),(12,14, "QTY"),(15,18, "PART"),(19,26, "DESCRIPTION"),(27,29, "QTY"),(30,33, "PART"),(34,45, "DESCRIPTION"),(46,48, "QTY"),(49,52, "PART"),(53,63, "DESCRIPTION"),(64,66, "QTY"),
                     (67,70, "PART"),(71,82, "DESCRIPTION"),(83,85, "QTY"),(86,89, "PART"),(90,99, "DESCRIPTION"),(100,102, "QTY"),]}
   ),
  (
       "1. COTTER PIN - x1 (AS PER SAMPLE) 2. ASSY 601 P4 - CAP SREWS - x8 3. ASSY 601 P3 - WASHERS - x8 4. ASSY 401 P6 - CAP SCREWS - x8 5. ASSY-1101 P1 SLING L2 1190mm WITH HOOKS AND SAFETY LATCHES ON BOTH ENDS",
       {"entities": [(3,13, "DESCRIPTION"),(16,18, "QTY"),(38,49, "PART"),(52,61, "DESCRIPTION"),(64,66, "QTY"),(70,81, "PART"),(84,91, "DESCRIPTION"),(94,96, "QTY"),(100,111, "PART"),(114,124, "DESCRIPTION"),(127,129, "QTY"),(133,145, "PART"),(146,204, "DESCRIPTION"),]}
   ),
  (
       "full refurbisment of the tool including: * Change of oil filters * Change of the hoses * Change of electric switches * Change of pump orings and sealings * Complete cleaning of pump and valves * Painting of frame and pump",
       {"entities": [(0,221, "DESCRIPTION"),]}
   ),
  (
       "1. SLING ASSY P56 = x1. 2. HOIST RING BASE P47 ON ASSY -30 x2 3. HOIST RING P48 ON ASSY-30 x2",
       {"entities": [(3,13, "DESCRIPTION"),(14,17, "PART"),(20,22, "QTY"),(27,42, "DESCRIPTION"),(43,46, "PART"),(47,58, "DESCRIPTION"),(59,61, "QTY"),(65,75, "DESCRIPTION"),(76,79, "PART"),(80,90, "DESCRIPTION"),(91,93, "QTY"),]}
   ),
  (
       "1- Integration of variant 98V71003002002 2- Upgrade to index J",
       {"entities": [(0,40, "DESCRIPTION"),]}
   ),
  (
       "1-Hose assemblies damages to be replaced 2-Handpump assembly to be replaced 3- Straight and elbow adapter at bottow of the filters damaged: to be replaced 4-Filler/breather cap to be replaced 5-Gauge damaged: to be replaced",
       {"entities": [(0,17, "DESCRIPTION"),(41,60, "DESCRIPTION"),(79,130, "DESCRIPTION"),(155,176, "DESCRIPTION"),(192,207, "DESCRIPTION"),]}
   ),
  (
       "0-0-364-68 x1 GN319-KT-20-M5-C x1",
       {"entities": [(0,10, "PART"),(11,13, "QTY"),(14,30, "PART"),(31,33, "QTY"),]}
   ),
  (
       "-8 screws -4 wheels",
       {"entities": [(0,2, "PART"),(3,9, "DESCRIPTION"),(10,12, "PART"),(13,19, "DESCRIPTION"),]}
   ),
  (
       "1. Pan Head Screw P4-40 1/4 Lg SS P18 NAS600-4P- x1.",
       {"entities": [(3,17, "DESCRIPTION"),(18,23, "PART"),(34,37, "PART"),(49,51, "QTY"),]}
   ),
  (
       "1. Valve Knob- P4 2. Connector -9 3. Metal cap -34 4. Air chuck -6",
       {"entities": [(3,14, "DESCRIPTION"),(15,17, "PART"),(21,30, "DESCRIPTION"),(31,33, "PART"),(37,46, "DESCRIPTION"),(47,50, "PART"),(54,63, "DESCRIPTION"),(64,66, "PART"),]}
   ),
  (
       "1. Streamer to be changed",
       {"entities": [(3,11, "DESCRIPTION"),]}
   ),
  (
       "1- P31 x2 2- P39 x1 3- P52 x1",
       {"entities": [(3,6, "PART"),(7,9, "QTY"),(13,16, "PART"),(17,19, "QTY"),(23,26, "PART"),(27,29, "QTY"),]}
   ),
  (
       "1- Spinning laser x1 2- Battery NiMH x6",
       {"entities": [(3,17, "DESCRIPTION"),(18,20, "QTY"),(24,36, "DESCRIPTION"),(37,39, "QTY"),]}
   ),
  (
       "P42 - FORCE GAUGE",
       {"entities": [(0,3, "PART"),(6,17, "DESCRIPTION"),]}
   ),
  (
       "1. Damaged connector 1QF2-3S64C replaced",
       {"entities": [(11,20, "DESCRIPTION"),(21,31, "PART"),]}
   ),
  (
       "M10 X 20 SCREW FOR MOUNTING PAD",
       {"entities": [(0,3, "PART"),(4,8, "QTY"),(9,31, "DESCRIPTION"),]}
   ),
  (
       "To be replaced: 1- P7 : x4 2- P2 P15 P16: x1 3- P11.2 x6",
       {"entities": [(19,21, "PART"),(24,26, "QTY"),(30,32, "PART"),(33,36, "PART"),(37,40, "PART"),(42,44, "QTY"),(48,53, "PART"),(54,56, "QTY"),]}
   ),
  (
       "Items to be replaced: -123 -114 -95 -145 -120 -124",
       {"entities": [(22,26, "PART"),(27,31, "PART"),(32,35, "PART"),(36,40, "PART"),(41,45, "PART"),(46,50, "PART"),]}
   ),
  (
       "1. REPLACED ANTI SLIP PADS. 2. T-BALL LOCK PIN 5/8 inch x4.5 3. HANDLE PUMP PN:H-1009. 01",
       {"entities": [(12,26, "DESCRIPTION"),(31,55, "DESCRIPTION"),(56,60, "QTY"),(64,75, "DESCRIPTION"),(79,89, "PART"),]}
   ),
  (
       "MS21915J6-4 x1 MS21902J4 x1 MS21913J4 x1 MS21913J5 x1 MS21914-4J x1 MS21916J5-4 x1 -P/N-104, hydraulic hose to be change -P/N-108, thread damaged + (4. 4FMTXSS / 1870. 14-14) 5 streamers",
       {"entities": [(0,11, "PART"),(12,14, "QTY"),(15,24, "PART"),(25,27, "QTY"),(28,37, "PART"),(38,40, "QTY"),(41,50, "PART"),(51,53, "QTY"),(54,64, "PART"),(65,67, "QTY"),(68,79, "PART"),(80,82, "QTY"),
                     (83,91, "PART"),(93,107, "DESCRIPTION"),(121,129, "PART"),(131,137, "DESCRIPTION"),(149,159, "PART"),(162,173, "PART"),]}
   ),
  (
       "1. Shoulder Screw 1/4 inchx20 - x2.",
       {"entities": [(3,17, "DESCRIPTION"),(32,34, "QTY"),]}
   ),
  (
       "90128A628 , 91259A716 ,3777T12 , 3827T36",
       {"entities": [(0,9, "PART"),(12,21, "PART"),(23,30, "PART"),(33,40, "PART"),]}
   ),
  (
       "1. New Foam set to manufacture 2- All assy 8 missing except P26: to be manufactured 3- 4 Y2004 damaged + 1 missing:5 new to supply 4-1 MS3181-10RAL-6 (cache prise) to supply",
       {"entities": [(87,94, "PART"),(135,149, "PART"),]}
   ),
  (
       "Hose Assy -60 x1 Cable assy -96 x1 Casters -117 x4",
       {"entities": [(0,4, "DESCRIPTION"),(5,13, "PART"),(14,16, "QTY"),(17,22, "DESCRIPTION"),(23,31, "PART"),(32,34, "QTY"),(35,42, "DESCRIPTION"),(43,47, "PART"),(48,50, "QTY"),]}
   ),
  (
       "2 x 95680A210P504 Non accurate HAND LEVERS to be replaced by BALL LOCK PIN",
       {"entities": [(4,17, "PART"),]}
   ),
  (
       "1-2xplastic pads -208 2-2xquick release pins 3-2xspring plungers 4-1xsocket -542",
       {"entities": [(17,21, "PART"),(76,80, "PART"),]}
   ),
  (
       "1. CLUTCH SPANNER PN: 98F07203020208",
       {"entities": [(3,17, "DESCRIPTION"),(22,36, "PART"),]}
   ),
  (
       "P1 P/N 98F71201001106 GSE ADAPTOR x1 P13 K1'1/16B SOCKET x1",
       {"entities": [(0,2, "PART"),(7,21, "PART"),(22,33, "DESCRIPTION"),(34,36, "QTY"),(37,40, "PART"),(50,56, "DESCRIPTION"),(57,59, "QTY"),]}
   ),
  (
       "1. Pressure Regulator -4 = x1. 2. Air Chuck Valve -6 = x1 3. Cross fitting -38 = x1 4. Bleed Valve -40 = x1 5. Nipple -8 = x1",
       {"entities": [(3,21, "DESCRIPTION"),(22,24, "PART"),(27,29, "QTY"),(34,49, "DESCRIPTION"),(50,52, "PART"),(55,57, "QTY"),(61,74, "DESCRIPTION"),(75,78, "PART"),(81,83, "QTY"),
                     (87,98, "DESCRIPTION"),(99,102, "PART"),(105,107, "QTY"),(111,117, "DESCRIPTION"),(118,120, "PART"),(123,125, "QTY"),]}
   ),

  (
       "1. STORAGE BOX -22 = x1",
       {"entities": [(3,14, "DESCRIPTION"),(15,18, "PART"),(21,23, "QTY"),]}
   ),
  (
       "09 -COTTER PIN x1 02 -CABLE RETAINER",
       {"entities": [(0,2, "PART"),(3,14, "DESCRIPTION"),(15,17, "QTY"),(18,20, "PART"),(21,36, "DESCRIPTION"),]}
   ),
  (
       "Stop valve -721",
       {"entities": [(0,10, "DESCRIPTION"),(11,15, "PART"),]}
   ),
  (
       "P15 - WASHER x4 P4 - WASHER RETAINING x4 P3 - BOLT HEX HEAD x4 P16 - LOCTITE. 480 = x1",
       {"entities": [(0,3, "PART"),(6,12, "DESCRIPTION"),(13,15, "QTY"),(16,18, "PART"),(21,37, "DESCRIPTION"),(38,40, "QTY"),(41,43, "PART"),(46,59, "DESCRIPTION"),(60,62, "QTY"),
                     (63,66, "PART"),(69,81, "DESCRIPTION"),(84,86, "QTY"),]}
   ),
  (
       "2 hex screws (CSTDVIS0889) 1 bolt index (WDS : 826-201)",
       {"entities": [(2,12, "DESCRIPTION"),(14,25, "PART"),(29,39, "DESCRIPTION"),(47,54, "PART"),]}
   ),
  (
       "1. POWER CABLE P2 = x1.(MISSING) 2. TEST CABLES AND TEST PROBES ASSY -3 = 1set (DAMAGED) 3. HEAVY DUTY TEST CABLES ASSY -4 = 1set (DAMAGED) 4. BATTERY = x1 (DAMAGED)",
       {"entities": [(15,17, "PART"),(64,71, "PART"),(115,122, "PART"),(143,150, "DESCRIPTION"),(153,155, "QTY"),]}
   ),
  (
       "STAMP PART MISSING ON ASSY -046",
       {"entities": [(0,31, "DESCRIPTION"),]}
   ),
  (
       "P10 x2 SCREWS MODIFED 11 x2 THUMB NUT 9 x1 VACUUM CUP 6 x1 INSERT 7 x1 PLATE.",
       {"entities": [(0,3, "PART"),(4,6, "QTY"),(7,13, "DESCRIPTION"),(22,24, "PART"),(25,27, "QTY"),(28,37, "DESCRIPTION"),(38,39, "PART"),(40,42, "QTY"),(43,53, "DESCRIPTION"),
                     (54,55, "PART"),(56,58, "QTY"),(59,65, "DESCRIPTION"),(66,67, "PART"),(68,70, "QTY"),(71,76, "DESCRIPTION"),]}
   ),
  (
       "1. PROTECT 2 P060 x2 2. PROTECT P058 = x1. 3. HOISTING POINT P718 ON ASSY 102 = x1. 4. HOISTING POINT P254 ON ASST 116 = x1.",
       {"entities": [(3,10, "DESCRIPTION"),(13,17, "PART"),(18,20, "QTY"),(24,31, "DESCRIPTION"),(32,36, "PART"),(39,41, "QTY"),(46,60, "DESCRIPTION"),(61,65, "PART"),(66,77, "DESCRIPTION"),(80,82, "QTY"),
                     (87,101, "DESCRIPTION"),(102,106, "PART"),(107,118, "DESCRIPTION"),(121,123, "QTY"),]}
   ),
  (
       "1. D SHACKLE x2 PN: DIN 82101 M10",
       {"entities": [(3,12, "DESCRIPTION"),(13,15, "QTY"),(20,33, "PART"),]}
   ),
  (
       "500. CATHETER x1 505. HOSE CLAMP x3 504. VALVE x1 042. WARNING STREAMER,COMPL",
       {"entities": [(0,3, "PART"),(5,13, "DESCRIPTION"),(14,16, "QTY"),(17,20, "PART"),(22,32, "DESCRIPTION"),(33,35, "QTY"),(36,39, "PART"),(41,46, "DESCRIPTION"),(47,49, "QTY"),
                     (50,53, "PART"),(55,77, "DESCRIPTION"),]}
   ),
  (
       "1) 82 - WASHER FORM A x2 2) 50 - SPECIAL WASHER x4 3) 87 - LOCKING PIN x4 4) 55 - WASHER SPECIAL x4 5) 54 - PIN LOCK x2 6) 92 - LANYARD CABLE",
       {"entities": [(3,5, "PART"),(8,21, "DESCRIPTION"),(22,24, "QTY"),(28,30, "PART"),(33,47, "DESCRIPTION"),(48,50, "QTY"),(54,56, "PART"),(59,70, "DESCRIPTION"),(71,73, "QTY"),
                     (77,79, "PART"),(82,96, "DESCRIPTION"),(97,99, "QTY"),(103,105, "PART"),(108,116, "DESCRIPTION"),(117,119, "QTY"),(123,125, "PART"),(128,141, "DESCRIPTION"),]}
   ),
  (
       "98L20003001262 CENTREUR = 2 98L20003001264 MOLETTE = 1 98L20003001236 BAGUE = 1 CCSTDVIS0312 SCREW CHC = 2",
       {"entities": [(0,14, "PART"),(15,23, "DESCRIPTION"),(28,42, "PART"),(43,50, "DESCRIPTION"),(55,69, "PART"),(70,75, "DESCRIPTION"),(80,92, "PART"),(93,102, "DESCRIPTION"),]}
   ),
  (
       "KNOB - M8 x2.",
       {"entities": [(0,4, "DESCRIPTION"),(7,9, "PART"),(10,12, "QTY"),]}
   ),
  (
       "ASSY 042 P712- DIAL INDICATOR x1 ASSY 106 P218- PROTECT x4 ASSY 102 P701- BALL LOCK PIN x1, P703 -LANYARD x2 ASSY 100 P900- WARNING STREAMER x1 ASSY 300 P990- USER GUIDE",
       {"entities": [(0,14, "PART"),(15,29, "DESCRIPTION"),(30,32, "QTY"),(33,47, "PART"),(48,55, "DESCRIPTION"),(56,58, "QTY"),(59,73, "PART"),(74,87, "DESCRIPTION"),(88,90, "QTY"),
         (92,96, "PART"),(97,105, "DESCRIPTION"),(106,108, "QTY"),(109,123, "PART"),(124,140, "DESCRIPTION"),(141,143, "QTY"),(144,158, "PART"),(159,169, "DESCRIPTION"),]}
   ),
  (
       "P27 x2 (ASSY -042) P24 x2 (ASSY -042) P25 x2 (ASSY -042) P28 x1 (ASSY -042)",
       {"entities": [(0,3, "PART"),(4,6, "QTY"),(19,22, "PART"),(23,25, "QTY"),(38,41, "PART"),(42,44, "QTY"),(57,60, "PART"),(61,63, "QTY"),]}
   ),
  (
       "2. qty of Pull Handle, 4 Nos. 9C3051P09, 4 Nos. 9C3051P10, 4 Nos. Cap Screw, Warning streamer.",
       {"entities": [(30,39, "PART"),(48,57, "PART"),]}
   ),
  (
       "1. KNURLED SCREW P500 - x1 2. THREADED PIN ASSY -108 - x1",
       {"entities": [(3,16, "DESCRIPTION"),(17,21, "PART"),(24,26, "QTY"),(30,42, "DESCRIPTION"),(43,52, "PART"),(55,57, "QTY"),]}
   ),
  (
       "2 rubber plates -476 1 knurled screw -484 1 captive washer -779 2 thrust piece (NLM07140-16) 1 insert -482 broken",
       {"entities": [(2,15, "DESCRIPTION"),(16,20, "PART"),(23,36, "DESCRIPTION"),(37,41, "PART"),(44,58, "DESCRIPTION"),(59,63, "PART"),(66,78, "DESCRIPTION"),(80,88, "PART"),(95,101, "DESCRIPTION"),(102,106, "PART"),]}
   ),
  (
       "P706 STRAP -25mm x1",
       {"entities": [(0,4, "PART"),(17,19, "QTY"),]}
   ),
  (
       "3 cotter pins replaced 3 flags NAS1756-24 replaced",
       {"entities": [(31,41, "PART"),]}
   ),
  (
       "Spring Pin P708 = x2,520 = x2",
       {"entities": [(0,10, "DESCRIPTION"),(11,15, "PART"),(27,29, "QTY"),]}
   ),
  (
       "1. TYGON TUBING P47, P56 2. DUST CAP P19, P24 x6 EACH 3. HOSE CLAMP P54 = x1. 4. WARNING STREAMER P88 x6 5. QD COUPLER P29, P31, P39 x2 Each 6. DUST PLUG P32, P22, P40 x3 Each 7. O-RING P26, P34 x2 Each'",
       {"entities": [(3,15, "DESCRIPTION"),(16,19, "PART"),(21,24, "PART"),(28,36, "DESCRIPTION"),(37,40, "PART"),(42,45, "PART"),(46,48, "QTY"),(57,67, "DESCRIPTION"),(68,71, "PART"),(74,76, "QTY"),
                     (81,97, "DESCRIPTION"),(98,101, "PART"),(102,104, "QTY"),(108,118, "DESCRIPTION"),(119,122, "PART"),(124,127, "PART"),(129,132, "PART"),(133,135, "QTY"),(144,153, "DESCRIPTION"),(154,157, "PART"),(159,162, "PART"),(164,167, "PART"),(168,170, "QTY"),
                     (179,185, "DESCRIPTION"),(186,189, "PART"),(191,194, "PART"),(195,197, "QTY"),]}
   ),
  (
       "1. WARNING STREAMER NAS1756-24 x2 FOC 2. CABLE ASSY x2 FOC",
       {"entities": [(3,19, "DESCRIPTION"),(20,30, "PART"),(31,33, "QTY"),(41,51, "DESCRIPTION"),(52,54, "QTY"),]}
   ),
  (
       "P7 FUSE PIN x2 -32 30800-0017 x1",
       {"entities": [(0,2, "PART"),(3,11, "DESCRIPTION"),(12,14, "QTY"),(15,29, "PART"),(30,32, "QTY"),]}
   ),
  (
       "C88-8-82 x1",
       {"entities": [(0,8, "PART"),(9,11, "QTY"),]}
   ),
  (
       "CL -5-BLPL-1. 50",
       {"entities": [(0,16, "PART"),]}
   ),
  (
       "1 ball lock pin - (CL -5-BLPL-1. 50)",
       {"entities": [(2,15, "DESCRIPTION"),(19,35, "PART"),]}
   ),
  (
       "P111 - EXTENSION POLE 16'' = x1 P115 - BALL LOCK PIN = x1 P119 -SOC HEAD CAP SCREW x1 P120 -FLAT HEAD SOCKET SCREW = x1 P121- FLAT HEAD SOCKET SCREW = x2",
       {"entities": [(0,4, "PART"),(7,21, "DESCRIPTION"),(29,31, "QTY"),(32,36, "PART"),(39,52, "DESCRIPTION"),(55,57, "QTY"),(58,62, "PART"),(63,82, "DESCRIPTION"),(83,85, "QTY"),(86,90, "PART"),(91,114, "DESCRIPTION"),(117,119, "QTY"),(120,125, "PART"),(126,148, "DESCRIPTION"),(151,153, "QTY"),]}
   ),
  (
       "1] KW -20-1 = ROD END.RH = x1 2] -9 = JAM NUT.RH = x1 3] CL -21-KA-14. 0. LR = CABLE ASSY = x1",
       {"entities": [(3,11, "PART"),(14,24, "DESCRIPTION"),(27,29, "QTY"),(33,35, "PART"),(38,48, "DESCRIPTION"),(51,53, "QTY"),(57,76, "PART"),(79,89, "DESCRIPTION"),(92,94, "QTY"),]}
   ),
  (
       "ITEM #: KG -20-1 ROD END. LH = x1. P9 JAM NUT, LH x2 P10 JAM NUT, RH = x1.",
       {"entities": [(8,16, "PART"),(17,28, "DESCRIPTION"),(31,33, "QTY"),(35,37, "PART"),(38,49, "DESCRIPTION"),(50,52, "QTY"),(53,56, "PART"),(57,68, "DESCRIPTION"),(71,73, "QTY"),]}
   ),
  (
       "P2 - RUBBER U CHANNEL -636-4250_21GSE_D_DET-201-18 = x1.",
       {"entities": [(0,2, "PART"),(5,50, "DESCRIPTION"),(53,55, "QTY"),]}
   ),
  (
       "P2 P/N = H -1009-01 ASSEMBLY HANDLE = x1 P29 HOSE P30 Adapter, Expanding Pipe = x1 P36 Blowgun = x1 P37 Valve, Plug = x1 P38 Caplug, Tapered = x1 P42 Ring, Ram Protection = x1",
       {"entities": [(9,19, "PART"),(20,35, "DESCRIPTION"),(38,40, "QTY"),(41,44, "PART"),(45,49, "DESCRIPTION"),(50,53, "PART"),(54,77, "DESCRIPTION"),(80,82, "QTY"),(83,86, "PART"),(87,94, "DESCRIPTION"),(97,99, "QTY"),(100,103, "PART"),(104,115, "DESCRIPTION"),(118,120, "QTY"),
                     (121,124, "PART"),(125,140, "DESCRIPTION"),(143,145, "QTY"),(146,149, "PART"),(150,170, "DESCRIPTION"),(173,175, "QTY"),]}
   ),
  (
       "P66 -89ASSY LONG HAND KNOB ASSY x2 P13 -89ASSY CLAMP PLATE x2 P74 -89ASSY SHORT HAND KNOB ASSY x2",
       {"entities": [(0,3, "PART"),(4,31, "DESCRIPTION"),(32,34, "QTY"),(35,38, "PART"),(39,58, "DESCRIPTION"),(59,61, "QTY"),(62,65, "PART"),(66,94, "DESCRIPTION"),(95,97, "QTY"),]}
   ),

  (
       "1. ADAPTER-SLAT PCU x2 2. ADAPTER-FLAP PCU x2 3. CABLE ASSY = x1 4. PULLEY P03 x2 5. PULLEY GUARD P04 x2 6. WHEEL ROLLER NYLATRON 05 x2 7. MOVABLE MAST P02 x3 8. SAFETY BRAKE ASSY P10G x3 9. CARRIAGE, SL ADVANTAGE = x1.",
       {"entities": [(3,19, "DESCRIPTION"),(20,22, "QTY"),(26,42, "DESCRIPTION"),(43,45, "QTY"),(49,59, "DESCRIPTION"),(62,64, "QTY"),(68,74, "DESCRIPTION"),(75,78, "PART"),(79,81, "QTY"),(85,97, "DESCRIPTION"),(98,101, "PART"),(102,104, "QTY"),(108,129, "DESCRIPTION"),(130,132, "PART"),(133,135, "QTY"),
                     (139,151, "DESCRIPTION"),(152,155, "PART"),(156,158, "QTY"),(162,179, "DESCRIPTION"),(180,184, "PART"),(185,187, "QTY"),(191,213, "DESCRIPTION"),(216,218, "QTY"),]}
   ),
  (
       "FOR x31 IN ASSY -41: C78033-44: INBOARD ARM. MANUFACTURING. x1 IN ASSY -42: C78033-45: OUTBOARD ARM. MANUFACTURING. x1",
       {"entities": [(0,19, "DESCRIPTION"),(21,30, "PART"),(32,43, "DESCRIPTION"),(60,62, "QTY"),(63,74, "DESCRIPTION"),(76,85, "PART"),(87,99, "DESCRIPTION"),(116,118, "QTY"),]}
   ),
  (
       "MISSING: 81-12-200-16 x4 81-46-101-41 x8 81-41-102-24 x4",
       {"entities": [(9,21, "PART"),(22,24, "QTY"),(25,37, "PART"),(38,40, "QTY"),(41,53, "PART"),(54,56, "QTY"),]}
   ),
  (
       "NEEDS REPLACEMENT BATTERY (CR2032) x1",
       {"entities": [(18,25, "DESCRIPTION"),(27,33, "PART"),(35,37, "QTY"),]}
   ),
  (
       "NEW BATTERY FOR 956A3209P08 (GAUGE) x1",
       {"entities": [(16,27, "PART"),(36,38, "QTY"),]}
   ),
  (
       "IAS -2081-01 x1 IAS -4030-06 x1 IAS -3090-01 x1 IAS -2990-01 x1 IAS -3110-01 x1 IAS -3160-01 x1",
       {"entities": [(0,12, "PART"),(13,15, "QTY"),(16,28, "PART"),(29,31, "QTY"),(32,44, "PART"),(45,47, "QTY"),(48,60, "PART"),(61,63, "QTY"),(64,76, "PART"),(77,79, "QTY"),(80,92, "PART"),(93,95, "QTY"),]}
   ),

  (
       "-P73 ROPE E.2 LG:320mm (12. 6) x1 -P75 ROPE E.2 LG:250mm (10) x1",
       {"entities": [(0,4, "PART"),(31,33, "QTY"),(34,38, "PART"),(62,64, "QTY"),]}
   ),
  (
       "DYNAMOMETER FOUND WITH A SMALL DENT INSIDE THE GAUGE DIAL FACE . 00",
       {"entities": [(0,67, "DESCRIPTION"),]}
   ),
  (
       "MISSING MASTER LINK A -342-1/2W x1 . 00",
       {"entities": [(20,31, "PART"),(32,34, "QTY"),]}
   ),
  (
       "UNIT ARRIVED WITH: MISSING -29 ASSY x1 MISSING -33 ASSY x2 RE-PASTE LOOSE ASSYS -33, -85, -31",
       {"entities": [(27,30, "PART"),(36,38, "QTY"),(47,50, "PART"),(56,58, "QTY"),(62,79, "DESCRIPTION"),(80,83, "PART"),(85,88, "PART"),(90,93, "PART"),]}
   ),
  (
       "P/N: 856A3707P24 P/N: 856A3707P23 TO BE REPLACED",
       {"entities": [(5,16, "PART"),(22,33, "PART"),]}
   ),

  (
       "ORIGINAL PO T009587-11 P/N: C71029-6 S/N: MCC180046-1-1, MCC180046-1-2 ORIGINAL PO T009581-12 P/N: C72019-11 S/N: MCF1800084-7 ORIGINAL PO T009588-11 P/N: C71029-16 S/N: MCF1800084-17",
       {"entities": [(28,36, "PART"),(99,108, "PART"),(155,164, "PART"),]}
   ),
  (
       "UNIT MISSING: AN919-10K REDUCER x1 AN919-6K REDUCER x1 5-BTX-SS NUT x2",
       {"entities": [(14,23, "PART"),(24,31, "DESCRIPTION"),(32,34, "QTY"),(35,43, "PART"),(44,51, "DESCRIPTION"),(52,54, "QTY"),(55,63, "PART"),(64,67, "DESCRIPTION"),(68,70, "QTY"),]}
   ),

  (
       "UNIT MISSING: 5-BTX-SS NUT x2 5-4-TRBTX-SS TUBE END REDUCER x2 AN919-10K REDUCER x1",
       {"entities": [(14,22, "PART"),(23,26, "DESCRIPTION"),(27,29, "QTY"),(30,42, "PART"),(43,59, "DESCRIPTION"),(60,62, "QTY"),(63,72, "PART"),(73,80, "DESCRIPTION"),(81,83, "QTY"),]}
   ),
  (
       "DAMAGED PART: DIAL (BENT) x1 CAPACITY 5000lb DIVISIONS 50 LB",
       {"entities": [(0,60, "DESCRIPTION"),]}
   ),
  (
       "DAMAGED: IN ASSY C7102911 -C71029-64 (AFT DYNAMOMETER) INNER FACE FLOATING NOTE: C71029-11 NEEDS TO BE FLIPPED IT IS ON BACKWARDS",
       {"entities": [(26,36, "PART"),(38,53, "DESCRIPTION"),]}
   ),
  (
       "DAMAGED GAUGE (01930134) TESA IP65 S/N: 13423054 / FAILED REPLACED BY GAUGE (01930261) TESA S/N: 3D104901",
       {"entities": [(15,23, "PART"),(77,85, "PART"),]}
   ),
  (
       "DAMAGED: P08 INDICATOR REF. PN 01930134 S/N:13423055 x1 *WILL BE REPLACED BY SAME P/N (01930134) S/N: 16473030 x1",
       {"entities": [(87,95, "PART"),]}
   ),
  (
      "UNIR ARRIVED MISSING: P85 FWD CRADLE PIN x1 P44 HEADED PIN x1 P94 BANANA BRACKET x1 TO COMPLETE -93 ASSY REQUIRES x1 OF EACH.",
      {"entities": [(22, 25, "PART"),(26,40, "DESCRIPTION"), (41, 43, "QTY"),(44, 47, "PART"),(48,58, "DESCRIPTION"), (59, 61, "QTY"),(62, 65, "PART"),(66,80, "DESCRIPTION"), (81, 83, "QTY"),(96, 99, "PART"), (114, 116, "QTY")]}
  ),
  (
       "P519HT PIPE = x1. 520 PIPE CLAMP x2 522 VIBRATION ABSORBER DGM x4 516 HT-PIPE MUFFLE x2 507 KNURLED HEAD SCREW x2 500 BALL LOCK PIN x2",
       {"entities": [(0,6, "PART"),(7,11, "DESCRIPTION"), (14, 16, "QTY"),(18,21, "PART"),(22,32, "DESCRIPTION"), (33, 35, "QTY"),(36,39, "PART"),(40,62, "DESCRIPTION"), (63, 65, "QTY"),(66,69, "PART"),(70,84, "DESCRIPTION"), (85, 87, "QTY"),
                     (88,91, "PART"),(92,110, "DESCRIPTION"), (111, 113, "QTY"),(114,117, "PART"),(118,131, "DESCRIPTION"), (132, 134, "QTY"),]}
   ),
  (
      "115 BALL LOCK PIN = x1 P111 16' EXTENSION POLE = x1",
      {"entities": [(0, 3, "PART"), (4, 17, "DESCRIPTION"), (20, 22, "QTY"),(23, 27, "PART"), (32, 46, "DESCRIPTION"), (49, 51, "QTY"),]}
  ),
 (
     "CQS -4_WEB (PROTECTIVE SLEEVE) x1 - RCV'D CQS -4_MID (PROTECTIVE SLEEVE) x1 - RCV'D CQS -4_FWD (PROTECTIVE SLEEVE) x1 - RCV'D MPLADEC0718 (SPECIAL PROTECTIVE COVER) x1 - RCV'D",
     {"entities": [(0, 10, "PART"), (12, 29, "DESCRIPTION"), (31, 33, "QTY"),(42, 52, "PART"), (54, 71, "DESCRIPTION"), (73, 75, "QTY"),(84, 94, "PART"), (96, 113, "DESCRIPTION"), (115, 117, "QTY"),(126, 137, "PART"), (139, 163, "DESCRIPTION"), (165, 167, "QTY"),]}
 ),
 (
     "K71001- 129 ,  K71001 -124 , K71001- 129 , BMB1712-5HP",
     {"entities": [(0, 11, "PART"),(15, 26, "PART"), (29, 40, "PART"),(43, 54, "PART"),]}
 ),
 (
     "P15 956A1066P15(PUSHER, HOLDER) x5",
     {"entities": [(0, 3, "PART"),(32, 34, "QTY"),]}
 ),
 (
     "PWC85967-1 TO PWC88111-1 P/N: CL -16-BLPL-3, 50-S-C x1 MODIFY FRAME P1 ADDING 2 NEW BUSHING PN: PWC88111- P1 - REP U x2",
     {"entities": [(0, 10, "PART"),(30, 43, "PART"),(45, 51, "PART"),(52, 54, "QTY"),(96, 108, "PART"),(117, 119, "QTY"),]}
 ),
 (
     "D12OUT00017PRT S/N: MCC180862-1 STORAGE BOXS NEEDS TO BE FIXED (DAMAGED FRAME,CORNER, LEGS, AND BUMPER)",
     {"entities": [(0, 14, "PART"),(15, 103, "DESCRIPTION"),]}
 ),
 (
     "MISSING OR DAMAGED: 220031 WICKE 225/83 WHEEL x2 P48 J 6332 121 MOUNT SHOCK ABSORBING x8 956A8600P166 ITEM-C SHACKLE SCREW x1 SEE THE COMPLETE LIST IN THE DESCRIPTION",
     {"entities": [(20, 26, "PART"),(27, 45, "DESCRIPTION"),(46, 48, "QTY"),(49, 63, "PART"),(64, 85, "DESCRIPTION"),(86, 88, "QTY"),(89, 101, "PART"),(109, 122, "DESCRIPTION"),(123, 125, "QTY"),]}
 ),
 (
     "759A PARTS KIT 1 75933 AIR PUMP KIT 1 76719 PIN 3 55923-150 EXT. SNAP RING 6 481-004 Y FILTER 1 55762-7 PUMP HANDLE 1 73318 SHIP ADAPTER 1 79595 RAIN HAT 1 NEW HYDRAULIC HOSES 1 MIL-PRF-5606 7",
     {"entities": [(0, 4, "PART"),(5, 14, "DESCRIPTION"),(17, 22, "PART"),(23, 35, "DESCRIPTION"),(38, 43, "PART"),(44, 47, "DESCRIPTION"),(50, 59, "PART"),(60, 74, "DESCRIPTION"),(77, 86, "PART"),(87, 93, "DESCRIPTION"),(96, 103, "PART"),(104, 115, "DESCRIPTION"),(118, 123, "PART"),(124, 136, "DESCRIPTION"),
                   (139, 144, "PART"),(145, 153, "DESCRIPTION"),(160, 175, "DESCRIPTION"),(178, 190, "PART"),]}
 ),
 (
     "WRONG PARTS: ON 98F71203001022: P10 REF. S232B AUGMENTER FACOM x1 P15 REF. K .15/16 inchB SOCKET FACOM x1",
     {"entities": [(32, 35, "PART"),(66, 69, "PART"),]}
 ),
 (
     "DAMAGED: 98D71203022208 JACK ( 13 TON ) x1 THE SECOND STAGE OF THE PISTON DISCHARGED P700 L02-100-40KFSTF CASTER D100 (ABC) x1 UPGRADE 98N7160H001-000 INLET HOISTING DEVICE MFG S/N: ACL0012 GE S/N: LEAP -001451",
     {"entities": [(9, 23, "PART"),(40, 42, "QTY"),(85, 105, "PART"),]}
 ),
 (
     "ADDING STRAPS TO SECURE B71040-66 IN CONTAINER",
     {"entities": [(24, 33, "PART"),]}
 ),
 (
     "Replace current casters w/LHF-ALTH 200K-AS [x24]",
     {"entities": [(24, 42, "PART"),(44, 47, "QTY"),]}
 ),
 (
     "WRONG PARTS: -SHACKLE NAS1044-5 x2 -SHACKLE NAS1042-5 x2 -SHACKLE NAS1042-4 x1 -SLING LINK NAS1049-8 x1",
     {"entities": [(22, 31, "PART"),(32, 34, "QTY"),(44, 53, "PART"),(54, 56, "QTY"),(66, 75, "PART"),(76, 78, "QTY"),(91, 100, "PART"),(101, 103, "QTY"),]}
 ),
 (
     "REF (P/N) x095052152A 2. 00 095052153A 1. 00 MC2448K49 4. 00 MC8863T12 1. 00 MC90128A277 1. 00 MC90967A109 1. 00 MC91831A011 1. 00 MC92125A250 1. 00 MC92917A140 1. 00 MC93625A250 1. 00 MC93935A335 1. 00 MC94500A236 1. 00 ",
     {"entities": [(10, 21, "PART"),(28, 38, "PART"),(45, 54, "PART"),(61, 70, "PART"),(77, 88, "PART"),(95, 106, "PART"),(113, 124, "PART"),(131, 142, "PART"),(149, 160, "PART"),(167, 178, "PART"),(185, 196, "PART"),(203, 214, "PART"),]}
 ),
 (
     "P7 T HANDLE QUICK RELEASE PIN 1/4 X 7/8 (90293A140) x2 P8 T HANDLE QUICK RELEASE PIN 5/16 X 1 (90293A211) x2 PREVENTATIVE MAINTENANCE",
     {"entities": [(0, 2, "PART"),(3, 11, "DESCRIPTION"),(52, 54, "QTY"),(55, 57, "PART"),(58, 66, "DESCRIPTION"),(106, 108, "QTY"),]}
 ),
 (
     "DAMAGED PARTS: 9471M34P03 CONNECTOR, J9 REPAIR PIN NUMBER -12",
     {"entities": [(15, 25, "PART"),(26, 35, "DESCRIPTION"),(37, 39, "PART"),(58, 61, "PART"),]}
 ),
 (
     "NOTE 2: -THE HORN MUST BE ADJUSTED TO 1500 daN 200 daN TO AVOID ANY STRUCTURE DAMAGES",
     {"entities": [(0, 85, "DESCRIPTION"),]}
 ),
 (
     "MISSING: ANA BL4 CABLE ANFAB x2",
     {"entities": [(9, 16, "PART"),(17, 28, "DESCRIPTION"),(29, 31, "QTY"),]}
 ),
 (
     "PN: 43/1936",
     {"entities": [(4, 11, "PART"),]}
 ),
 (
     "-12 06010-206 KNURLED NUT x2 -11 07120-06 PAD SCREW x1 -16 ISO8752-1, 5X16 SPRING PIN x1 -10 07140-06 PAD x2 -18 17M47 VACUUM HANDLE (SUCTION CUP HANDLE ONLY) x1",
     {"entities": [(0, 13, "PART"),(14, 25, "DESCRIPTION"),(26, 28, "QTY"),(29, 41, "PART"),(42, 51, "DESCRIPTION"),(52, 54, "QTY"),(55, 68, "PART"),(75, 85, "DESCRIPTION"),(86, 88, "QTY"),(89, 101, "PART"),(102, 105, "DESCRIPTION"),(106, 108, "QTY"),
                   (109, 118, "PART"),(119, 158, "DESCRIPTION"),(159, 161, "QTY"),]}
 ),
 (
     "ARRIVED MISSING: SHACKLE G -209-1 ARRIVED MISSING: AFT CRADLE PIN x1",
     {"entities": [(17, 24, "DESCRIPTION"),(25, 33, "PART"),(66, 68, "QTY"),]}
 ),
 (
     "UNIT MISSING BOLTS AN 4-12A x4 REBOND RUBBER PADS P23 x4",
     {"entities": [(19, 27, "PART"),(28, 30, "QTY"),(31, 49, "DESCRIPTION"),(50, 53, "PART"),(54, 56, "QTY"),]}
 ),
 (
     "UNIT MISSING LINK G -341-1/2",
     {"entities": [(18, 28, "PART"),]}
 ),
 (
     "UNIT ARRIVED WITH: GAUGES LOOSING MISSING P6589A12 BOX x1 MISSING LL2-K-6MS x2",
     {"entities": [(42, 50, "PART"),(55, 57, "QTY"),(66, 75, "PART"),(76, 78, "QTY"),]}
 ),
 (
     "-11 9568688P11(HANDLE) x1 -23 CL-.194-TAB-S (CABLE TABS) x2 -22 E.2 LG: 150MM (ROPE) x2 NEW BOX FOR TOOL",
     {"entities": [(0, 3, "PART"),(23, 25, "QTY"),(26, 43, "PART"),(45, 55, "DESCRIPTION"),(57, 59, "QTY"),(60, 63, "PART"),(64, 84, "DESCRIPTION"),(85, 87, "QTY"),]}
 ),
 (
     "MISSING P3 SHACKLE 43/874 (USE FOR THS ACTUATOR ONLY) x1 -MISSING P12 SHACKLE 91/724 (USE ON APU GTCP36-300 ONLY) x1 - MISSING BALL LOCK PIN SHEATH 43/1960. SHALL BE 1/4”X 3/4”",
     {"entities": [(8, 10, "PART"),(54, 56, "QTY"),(66, 69, "PART"),(114,116, "QTY"),]}
 ),

 (
     "CHANGE 9471M34P07 INDICATOR LED DAMAGED x1 CHANGE 9471M34P03 CONNECTOR, J9 DAMAGED x1 MISSING CAP FOR RECEPTACLE RJ45 CHANGE BATTERY.",
     {"entities": [(7, 17, "PART"),(18, 31, "DESCRIPTION"),(40, 42, "QTY"),(50, 60, "PART"),(61, 70, "DESCRIPTION"),(72, 74, "PART"),(83, 85, "QTY"),]}
 ),
 (
     "ASSY 2100 MANOMTER DAMAGED OR BATTERY DEAD x1 ASSY 2106 MISSING P01 x1 ASSY -2108 MISSING P02 x1",
     {"entities": [(0, 9, "PART"),(43, 45, "QTY"),(46, 55, "PART"),(64, 67, "PART"),(68, 70, "QTY"),(71, 81, "PART"),(90, 93, "PART"),(94, 96, "QTY"),]}
 ),
 (
     "CCHIHUG0037 AEROSHELL (1) DASSCHA0052P18 (2) DASSCHA0052P19 (1) CSTDANN0369 O'RING (1) CSTDANN0370 O'RING (1) CSTDANN0371 O'RING (1)",
     {"entities": [(0, 11, "PART"),(26, 40, "PART"),(45, 59, "PART"),(64, 75, "PART"),(87, 98, "PART"),(110, 121, "PART"),]}
 ),
 (
     "CCHIHUG0037 x4 SPHJOELT009370 C-H-JO-NUT008013 C-H-JO-RCL0019692 ORAR00017 OR4017200 CFCT150FCN7 CFCT110FCN7 DASSCHA0052P17 x1, P18 x2, P19 x1 SPMMLEV009266 DPTSAXE0010 x2 SPHPOENS009267-9 x1, -10 x1, -11 x2 , -14 x4 SPMPSJ0N008856",
     {"entities": [(0, 11, "PART"),(12, 14, "QTY"),(15, 29, "PART"),(30, 46, "PART"),(47, 64, "PART"),(65, 74, "PART"),(75, 84, "PART"),(85, 96, "PART"),(97, 108, "PART"),(109, 123, "PART"),(124, 126, "QTY"),(128, 131, "PART"),(132, 134, "QTY"),(136, 139, "PART"),(140, 142, "QTY"),(143, 156, "PART"),(157, 168, "PART"),
                   (169, 171, "QTY"),(172, 186, "PART"),(187, 188, "PART"),(189, 191, "QTY"),(193, 196, "PART"),(197, 199, "QTY"),(201, 204, "PART"),(205, 207, "QTY"),(210, 213, "PART"),(214, 216, "QTY"),(217, 231, "PART"),]}
 ),

 (
     "NEWS CONNECTOR HOSE ADAPTER x2 ASSY 401-P04 DAMAGED, HYDRAULIC HOSE x2 NEWS MOBIL JET OIL x3",
     {"entities": [(5, 27, "DESCRIPTION"),(28, 30, "QTY"),(31, 43, "PART"),]}
 ),
 (
     "DAMAGED WHEEL PUMP x1 JOINT POUR ALESAGE D 150 MOYEN x1 JOINT POUR ALESAGE D 110 MOYEN x1 SET OF BOLTS AND NUTS x1",
     {"entities": [(0, 114, "DESCRIPTION"),]}
 ),
 (
     "DAMAGED WHEEL PUMP x1 JOINT POUR ALESAGE D 150 MOYEN x1 JOINT POUR ALESAGE D 110 MOYEN x1 SET OF BOLTS AND NUTS x1",
     {"entities": [(0, 114, "DESCRIPTION"),]}
 ),
 (
     "CCHIHUG0044  ADD 16 CSTDVIS0852  MISSING 3 CSTDRON0217  MISSING 3 DASSCHA0052-P18 DAMAGE 2 DASSCHA0052-P19 DAMAGE 1 SPHPOENS009267-P14 DAMAGE 4 DASSCHA0055-10  DAMAGE 1 DASSCHA0055-11  DAMAGE 1 DASSCHA0055-07  DAMAGE 1 97045K55  DAMAGE 1 RBE11-6152",
     {"entities": [(0, 11, "PART"),(20, 31, "PART"),(43, 54, "PART"),(66, 81, "PART"),(91, 106, "PART"),(116, 134, "PART"),(144, 158, "PART"),(169, 183, "PART"),(194, 208, "PART"),(219, 227, "PART"),(238, 248, "PART"),]}
 ),
 (
     "D32AXJ00014G01 FLUID AEROSHELL 18 DASSCHA0052 -17 1 -18 2 -19 1 SPHPOENS009267 -10 1 -11 2",
     {"entities": [(0, 14, "PART"),(34, 45, "PART"),(64, 78, "PART"),]}
 ),
 (
     " ASSY 201 91251A588 (x2) 98370A019 (x4) 91240A031 (x2) 6 (x1)  3 (x2)(x1)  ASSY 301 2 (x1)  OIL (x4 liters) ASSY 401 3 (x2)  1 (x2)",
     {"entities": [(10, 19, "PART"),(21, 23, "QTY"),(25, 34, "PART"),(36, 38, "QTY"),(40, 49, "PART"),(51, 53, "QTY"),(55, 56, "PART"),(58, 60, "QTY"),(63, 64, "PART"),(84, 85, "PART"),(87, 89, "QTY"),(117, 118, "PART"),(120, 122, "QTY"),(125, 126, "PART"),(128, 130, "QTY"),]}
 ),
 (
     "mano 0 à 400bars, mano 0-250bars kit de joints flexible HLA-A-1659-1000",
     {"entities": [(0, 55, "DESCRIPTION"),(56, 71, "PART"),]}
 ),
 (
     "1/ Subassemblies 72 and 73 upgrade to 106 and 107. 2/ Subassembly 2 upgrade to 126 3/ Subassembly 93 upgrade to 112 4/ Subassemblies 81 upgrade to 127 5/ Subassemblies 92 upgrade to 128 6/ Marking",
     {"entities": [(17, 19, "PART"),(24, 26, "PART"),(27, 49, "DESCRIPTION"),(66, 67, "PART"),(68, 82, "DESCRIPTION"),(98, 100, "PART"),(101, 115, "DESCRIPTION"),(133, 135, "PART"),(136, 150, "DESCRIPTION"),(168, 170, "PART"),(171, 185, "DESCRIPTION"),]}
 ),
 (
     "UPGRADE TO K21010-54",
     {"entities": [(11, 20, "PART"),]}
 ),
 (
     "1/ Remove P856A3525P24. 2/ Add P856A3525P32, x1.",
     {"entities": [(10, 22, "PART"),(31, 43, "PART"),(45, 47, "QTY"),]}
 ),
 (
     "UPGRADE 956A8304G01---TO---G02",
     {"entities": [(8, 19, "PART"),]}
 ),
 (
     "1/ Upgrade RRT025893 to RRT025893-1",
     {"entities": [(11, 20, "PART"),(24, 35, "PART"),]}
 ),
 (
     "1/ Change bolts from P08 to P15 x6",
     {"entities": [(21, 24, "PART"),(28, 31, "PART"),(32, 34, "QTY"),]}
 ),

 (
     "Replace the P98F47103000220",
     {"entities": [(12, 27, "PART"),]}
 ),
 (
     "TRANSPORTATION FEE INCLUDED IN ZDSRP150045",
     {"entities": [(0, 42, "DESCRIPTION"),]}
 ),
 (
     "631-H/2 HAS TO BE CHANCHED",
     {"entities": [(0, 7, "PART"),]}
 ),
 (
     "CHANGE ONE PUMP: P/N DM00416",
     {"entities": [(21, 28, "PART"),]}
 ),
 (
     "UNIT MISSING P98V52008861200 WASHER x5",
     {"entities": [(13, 28, "PART"),(29, 35, "DESCRIPTION"),(36, 38, "QTY"),]}
 ),
 (
    "MVP BAG REMOVED FROM SN 240967-7-018 AND FITTED TO STAND SN 240967-7-011 (4 Mhr) FIT TRIAL MVP BAG TO 240967-7-018 AT RR LDC, INSTALL/REMOVE (5 Mhr & 3Mhr )",
    {"entities": [(0, 156, "DESCRIPTION"),]}
 ),
 (
    "JA0600621 JB0710621 JB0900881 Hydraulic oil KPL142",
    {"entities": [(0, 9, "PART"),(10, 19, "PART"),(20, 29, "PART"),(30, 43, "DESCRIPTION"),(44, 50, "PART"),]}
 ),
 (
    "1. FRICTION WASHERS UNMIT PRICE:760RMB/EA x2 2. SCREW M4 X 10 UNIT PRICE:2RMB/EA x5",
    {"entities": [(0, 83, "PART"),]}
 ),

 (
     "JA0950631 1 UN JA1500813 1 UN  DPTSBAN0028_A 2 UN  DPTSBAN0039_A 4 UN 23-511-200B25 3UN",
     {"entities": [(0, 9, "PART"),(15, 24, "PART"),(32, 45, "PART"),(53, 66, "PART"),(73, 86, "PART"),]}
 ),
 (
     "UNIT ARRIVED MISSING: P/N: 98D27303003:203 ARM P/N: 98D27303003.205 ARM",
     {"entities": [(27, 42, "PART"),(43, 46, "DESCRIPTION"),(52, 67, "PART"),(68, 71, "DESCRIPTION"),]}
 ),
 (
     "Shock mounts/bolts/wire lock - x8 Lanyards replacement Pin AM -90500-20L-U 1 - x1 Pin AM -91000-6BL - x2 Pin AM -91250-68L - x2 Stop nuts and screws Customer to order: OEM Caster Assy AM -1803-801 OEM Caster Assy AM -1803-701",
     {"entities": [(55, 58, "DESCRIPTION"),(59, 76, "PART"),(79, 81, "QTY"),(82,85, "DESCRIPTION"),(86, 99, "PART"),(102, 104, "QTY"),(105,108, "DESCRIPTION"),(109, 122, "PART"),(125, 127, "QTY"),(184, 196, "PART"),(213, 225, "PART"),]}
 ),
 (
     "1. IAE1F10363 29 ROPE UNIT PRICE:2262RMB/EA x2 2. IAE1F10363 34 ROPE UNIT PRICE:1511. 25RMB/EA x2",
     {"entities": [(3, 16, "PART"),(44, 46, "QTY"),(50, 63, "PART"),(95, 97, "QTY"),]}
 ),
 (
     "7L d’huile HVI 13",
     {"entities": [(0, 17, "DESCRIPTION"),]}
 ),
 (
     "REPLACEMENT PARTS REQUIRED: RP-.188X2.5-SXX ROLL PIN 1 MS16562-38 PIN-ROLL, .125 X 1. 000 x1 MS16562-73 SPIRAL PIN-ROLL x1 AN4-12A BOLT-MACHINE, HEX HEAD x1 MS21044N4 SELF-LOCKING NUT (REPLACES AN365-428A) x1 SEE FULL DESCRIPTION ABOVE",
     {"entities": [(28, 43, "PART"),(44, 52, "DESCRIPTION"),(55, 65, "PART"),(66, 74, "DESCRIPTION"),(90, 92, "QTY"),(93, 103, "PART"),(104, 119, "DESCRIPTION"),(120, 122, "QTY"),(123, 130, "PART"),(131, 153, "DESCRIPTION"),(154, 156, "QTY"),
                   (157, 166, "PART"),(167, 205, "DESCRIPTION"),(206, 208, "QTY"),]}
 ),
 (
     "6x bolts F/N20",
     {"entities": [(0, 14, "DESCRIPTION"),]}
 ),
 (
     "1ea hoist ring",
     {"entities": [(0, 3, "PART"),(4, 14, "DESCRIPTION"),]}
 ),
 (
     "2x Tube 856A1084P04 1x Tube 856A1084P51 1x Tube 856A1084P50 Oil recommended MIL -5606 aviation grade hydraulic oil with a capacity of approximately 1. 5 Gallons according to TDS Use 1010 Oil for pressure test according to drawing",
     {"entities": [(3, 7, "DESCRIPTION"),(8, 19, "PART"),(23, 27, "DESCRIPTION"),(28, 39, "PART"),(43, 47, "DESCRIPTION"),(48, 59, "PART"),(60, 229, "DESCRIPTION"),]}
 ),
 (
     "DAMAGED: IN ASSY 98V52008861046: P507 KKA -6-62030 COMBINATION ADAPTOR x1",
     {"entities": [(33, 37, "PART"),(38, 50, "PART"),(51, 70, "DESCRIPTION"),(71, 73, "QTY"),]}
 ),
 (
     "PN: K -1050",
     {"entities": [(0, 11, "PART"),]}
 ),
 (
     "Protection Flange [x1] Kit, Shaft Seal and O-Ring [x1] Hydraulic Coupling Kit (HRT -01) [x1] Hose Assembly, HRT -01 Return (16PE) [x1] Hose Assembly, HRT -01 Pressure (12PE) [x1] Calibration - RAT Test Tool [x1]",
     {"entities": [(0, 211, "DESCRIPTION"),]}
 ),
 (
     "PF51-003-1 TEARDOWN AND EVALUATE S/N 233066 [x1] REPLACEMENT PARTS REQUIRED: RP-.188X2.5-SXX ROLL PIN [x1] MS16562-38 PIN-ROLL, .125 X 1. 000 [x1] MS16562-73 SPIRAL PIN-ROLL [x1] SEE FULL DESCRIPTION ABOVE",
     {"entities": [(0, 10, "PART"),(45, 47, "QTY"),(77, 92, "PART"),(93, 101, "DESCRIPTION"),(103, 105, "QTY"),(107, 117, "PART"),(118, 126, "DESCRIPTION"),(143, 145, "QTY"),
                   (147, 157, "PART"),(158, 173, "PART"),(175, 177, "QTY"),]}
 ),
 (
     "MISSING OR DAMAGED: P90 C596-104B OR ALT. J -6332-208 SHOCK ABSORBING,MOUNT x8 (EXPIRED IN 2023) P117 SCREW CAP HEX HD STL-GRADE 5 ( .500-20 UNF) X 1. 000. LG x64 P132 WASHER FLAT STL-GRADE 5 OD (1. 06) X ID ( .53) X (.13) THK",
     {"entities": [(20, 23, "PART"),(24, 33, "PART"),(42, 53, "PART"),(97, 101, "PART"),(163, 167, "PART"),]}
 ),
 (
     "1. JAM NUT, 5/8-18 ELASTIC P8 x3 2. SPRING P10 x3 3. CASTER, STEM P12 x3 4. PUMP, HYDRAULIC HAND P16 = x1. 5. KIT SEAL REPLACEMENT = 1 SET PN: K -1050 6. HYDRAULIC FLUID = 1 GAL",
     {"entities": [(3, 26, "DESCRIPTION"),(27, 29, "PART"),(30, 32, "QTY"),(36, 42, "DESCRIPTION"),(43, 46, "PART"),(47, 49, "QTY"),(53, 65, "DESCRIPTION"),(66, 69, "PART"),(70, 72, "QTY"),(76, 96, "DESCRIPTION"),(97, 100, "PART"),(103, 105, "QTY"),(143, 152, "PART"),(154, 169, "DESCRIPTION"),]}
 ),
 (
     "P/N: AN4-12A x6 ( BOLT) P/N: 1/4-28 x6 ( NUT STEEL) -BOX",
     {"entities": [(5,12, "PART"),(13, 15, "QTY"),(18, 22, "DESCRIPTION"),(29,35, "PART"),(36, 38, "QTY"),(41, 50, "DESCRIPTION"),]}
 ),






]


## 5. Entraînement du modèle

In [11]:
# Un modèle spaCy vierge

nlp = spacy.blank("xx")

# Ajout pipeline NER
ner = nlp.add_pipe("ner")

# Ajout des étiquettes
labels = ["PART", "ACTION","DESCRIPTION", "OBS", "QTY"]
for label in labels:
    ner.add_label(label)

# Vérification des alignements (important pour SpaCy)
print("\n🔎 Vérification des alignements :")
for text, ann in TRAIN_DATA:
    doc = nlp.make_doc(text)
    tags = offsets_to_biluo_tags(doc, ann["entities"])
    print(text)
    print(tags)

# Créeation des objets d'exemples
examples = [Example.from_dict(nlp.make_doc(text), ann) for text, ann in TRAIN_DATA]

# 🏗️ Initialiser le modèle
nlp.initialize(get_examples=lambda: examples)

#  Entraînement du modèle
for i in range(20):
    losses = {}
    for batch in minibatch(examples, size=2):
        nlp.update(batch, losses=losses)
    print(f"📉 Iteration {i} - pertes : {losses}")


🔎 Vérification des alignements :
Ajout de la pièce P12GH-20-28-7 cassé avec une quantité x2
['U-ACTION', 'O', 'O', 'O', 'B-PART', 'I-PART', 'I-PART', 'I-PART', 'L-PART', 'U-OBS', 'O', 'O', 'O', 'U-QTY']
La pièce p12x3 a été removed car manquant
['O', 'O', 'U-PART', 'O', 'O', 'U-ACTION', 'O', 'U-OBS']
Fourniture et changement du joint REP021 fissuré x1
['O', 'O', 'U-ACTION', 'O', 'O', 'U-PART', 'U-OBS', 'U-QTY']
Suppression de la pièce P99X-44-90 manquant
['U-ACTION', 'O', 'O', 'O', 'B-PART', 'I-PART', 'L-PART', 'U-OBS']
S711V1001125 manque Item 53
['O', 'U-OBS', 'B-PART', 'L-PART']
 S711V1001231 Configuration label Item 004 unvisible: to replace
['O', 'O', 'O', 'O', 'B-PART', 'L-PART', 'U-OBS', 'O', 'O', 'U-ACTION']
 S711V1001232 Stop missing (x2): welding +painting rework
['O', 'O', 'O', 'U-OBS', 'O', 'U-QTY', 'O', 'O', 'O', 'O', 'O']
item 007 missing to be provided
['B-PART', 'L-PART', 'O', 'O', 'O', 'O']
S711V1001804 Hook bended: to be replaced
['O', 'B-DESCRIPTION', 'L-DESCRIPTION

## 6. Extration d'entité

In [12]:
test_text = "1. JAM NUT, 5/8-18 ELASTIC P8 x3 2. SPRING P10 x3 3. CASTER, STEM P12 x3 4. PUMP, HYDRAULIC HAND P16 = x1. 5. KIT SEAL REPLACEMENT = 1 SET PN: K -1050 6. HYDRAULIC FLUID = 1 GAL"
doc = nlp(test_text)

print("\n🔎 Résultat NER :")
for ent in doc.ents:
    print(f"{ent.text} → {ent.label_}")


🔎 Résultat NER :
JAM NUT, 5/8-18 ELASTIC → DESCRIPTION
P8 → PART
x3 → QTY
SPRING → DESCRIPTION
P10 → PART
x3 → QTY
CASTER, STEM → DESCRIPTION
P12 → PART
x3 → QTY
PUMP, HYDRAULIC HAND → DESCRIPTION
P16 → PART
x1 → QTY
K -1050 6 → PART
HYDRAULIC FLUID → DESCRIPTION


**Ici nous présentons un modèle NLP sans utilisation des Transformers, qui constitue une approche efficace pour traiter des textes non structurés.**

**Si vous souhaitez discuter de modèles basés sur les Transformers, n'hesitez pas à me contacter**